In [ ]:
print ('hola')

In [ ]:
# =====================================================================================
# CELDA 0. Historial de 12 meses para debutante y primera vez + registro de la enmienda
#   Se corre UNA vez, antes del cuaderno 16. No modifica la base: solo verifica y deja constancia.
#   Nueva definición (decisión del grupo, 11-oct-2026):
#     debutante_365   = el proveedor no tiene procesos en los 365 días anteriores (sin contar el día)
#     primera_vez_365 = el par proveedor-entidad no tiene procesos en esos 365 días
#   Se obtienen de columnas que la base ya trae (mismo universo y misma ventana que cuota y
#   recurrencia) y se comprueban recalculándolas desde cero con las fechas y los RUC.
# =====================================================================================
import os, json, hashlib
from datetime import datetime
import numpy as np
import pandas as pd
import duckdb

BASE = r"C:\SERCOP_FINAL\plan_A"
ENMIENDAS = r"C:\SERCOP_FINAL\enmiendas"
os.makedirs(ENMIENDAS, exist_ok=True)

def ruta(nombre):
    return os.path.join(BASE, nombre).replace("\\", "/")

def huella(path, n=16):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()[:n]

IX, PM = ruta("planA_indicadores.parquet"), ruta("planA_procesos_modelo.parquet")
con = duckdb.connect()
cols_ix = [r[0] for r in con.execute(f"DESCRIBE SELECT * FROM read_parquet('{IX}')").fetchall()]
cols_pm = [r[0] for r in con.execute(f"DESCRIBE SELECT * FROM read_parquet('{PM}')").fetchall()]
necesarias = ["ocid", "fecha_proceso", "entidad_ruc", "ind_proveedor_debutante",
              "ind_primera_vez_con_la_entidad", "ind_procesos_proveedor_12m", "ind_recurrencia_par_12m",
              "ind_cuota_prov_en_entidad_12m"]
faltan = [c for c in necesarias if c not in cols_ix]
if faltan:
    raise SystemExit(f"Faltan columnas en planA_indicadores: {faltan}. Pégame este mensaje.")
col_prov = "proveedor_ruc_final" if "proveedor_ruc_final" in cols_pm else None

# 1) Tabla de trabajo: fecha del proceso (día), RUC del proveedor y de la entidad, columnas de historial
con.execute(f"""
CREATE OR REPLACE TEMP TABLE t AS
SELECT i.ocid,
       CAST({'p.fecha_inicio_tender' if 'fecha_inicio_tender' in cols_pm else 'i.fecha_proceso'} AS DATE) AS dia,
       {'p.' + col_prov if col_prov else 'i.proveedor_ruc'} AS prov,
       i.entidad_ruc AS ent,
       CAST(i.ind_proveedor_debutante AS INTEGER)        AS deb_antes,
       CAST(i.ind_primera_vez_con_la_entidad AS INTEGER) AS pv_antes,
       i.ind_procesos_proveedor_12m AS n_prov_12m,
       i.ind_recurrencia_par_12m    AS n_par_12m,
       i.ind_cuota_prov_en_entidad_12m AS cuota_12m
FROM read_parquet('{IX}') i LEFT JOIN read_parquet('{PM}') p USING (ocid)
""")
n = con.execute("SELECT COUNT(*) FROM t").fetchone()[0]
print(f"Procesos: {n:,} | proveedor tomado de: {col_prov or 'proveedor_ruc (indicadores)'}")
print(con.execute("""SELECT COUNT(*) FILTER (WHERE dia IS NULL) AS sin_fecha,
                            COUNT(*) FILTER (WHERE prov IS NULL OR prov = '') AS sin_proveedor,
                            COUNT(*) FILTER (WHERE ent IS NULL OR ent = '') AS sin_entidad,
                            COUNT(*) FILTER (WHERE n_prov_12m IS NULL) AS sin_n_prov_12m,
                            COUNT(*) FILTER (WHERE n_par_12m IS NULL) AS sin_n_par_12m FROM t""").df().to_string(index=False))

# 2) Recalculo independiente: cuántos procesos tuvo el proveedor (y el par) en los 365 días previos
con.execute("""
CREATE OR REPLACE TEMP TABLE r AS
SELECT ocid,
  CASE WHEN prov IS NULL OR prov = '' THEN NULL ELSE COUNT(*) OVER (PARTITION BY prov ORDER BY dia
       RANGE BETWEEN INTERVAL 365 DAYS PRECEDING AND INTERVAL 1 DAY PRECEDING) END AS n_prov_rec,
  CASE WHEN prov IS NULL OR prov = '' OR ent IS NULL OR ent = '' THEN NULL ELSE COUNT(*) OVER
       (PARTITION BY prov, ent ORDER BY dia RANGE BETWEEN INTERVAL 365 DAYS PRECEDING AND INTERVAL 1 DAY PRECEDING) END AS n_par_rec
FROM t
""")
d = con.execute("SELECT t.*, r.n_prov_rec, r.n_par_rec FROM t JOIN r USING (ocid)").df()
d["dia"] = pd.to_datetime(d["dia"])      # mismo tipo de fecha en cualquier versión de DuckDB
d["anio"] = d["dia"].dt.year

# nuevas variables (desde las columnas de la base) y su versión recalculada
d["deb_365"] = np.where(d["n_prov_12m"].isna(), np.nan, (d["n_prov_12m"] == 0).astype(float))
d["pv_365"] = np.where(d["n_par_12m"].isna(), np.nan, (d["n_par_12m"] == 0).astype(float))
d["deb_365_rec"] = np.where(d["n_prov_rec"].isna(), np.nan, (d["n_prov_rec"] == 0).astype(float))
d["pv_365_rec"] = np.where(d["n_par_rec"].isna(), np.nan, (d["n_par_rec"] == 0).astype(float))

def coincide(a, b):
    ok = d[a].notna() & d[b].notna()
    return 100 * (d.loc[ok, a] == d.loc[ok, b]).mean(), int(ok.sum())

print("\n(A) Coherencia lógica (deben dar 0):")
c1 = int(((d["deb_antes"] == 1) & (d["n_prov_12m"] > 0)).sum())
c2 = int(((d["pv_antes"] == 1) & (d["n_par_12m"] > 0)).sum())
print(f"  debutante 'alguna vez' con procesos del proveedor en 12 meses: {c1:,}")
print(f"  primera vez 'alguna vez' con procesos del par en 12 meses:     {c2:,}")

print("\n(B) Columna de la base frente al recálculo desde fechas y RUC (% de coincidencia):")
cd, nd = coincide("deb_365", "deb_365_rec")
cp, npv = coincide("pv_365", "pv_365_rec")
print(f"  debutante_365:   {cd:.2f} % de {nd:,} procesos")
print(f"  primera_vez_365: {cp:.2f} % de {npv:,} procesos")
print("  Si no llega a 99 %, la base contó procesos de otro universo (por ejemplo catálogo) o usó otro borde de la ventana.")
print("  Diferencias en el conteo del proveedor (base menos recálculo), percentiles 1, 50 y 99:",
      np.nanpercentile(d["n_prov_12m"] - d["n_prov_rec"], [1, 50, 99]).round(1).tolist())

print("\n(C) Porcentaje de procesos por año: definición anterior ('alguna vez') frente a la nueva (12 meses)")
pre = d[d["dia"] < pd.Timestamp("2025-10-07")]
tab = pre.groupby("anio").agg(procesos=("ocid", "size"),
                              debutante_antes=("deb_antes", "mean"), debutante_365=("deb_365", "mean"),
                              primera_vez_antes=("pv_antes", "mean"), primera_vez_365=("pv_365", "mean"),
                              con_historial_par=("cuota_12m", lambda s: (s > 0).mean()))
for c in tab.columns[1:]:
    tab[c] = (100 * tab[c]).round(1)
tab.index = tab.index.astype(int)
print(tab.T.to_string())
tab.to_csv(os.path.join(ENMIENDAS, "enmienda_2026-10-16_tabla_por_anio.csv"))

desde = tab.loc[tab.index >= 2016]
rango_deb = (desde["debutante_365"].min(), desde["debutante_365"].max())
print(f"\n  debutante_365 entre 2016 y 2025: de {rango_deb[0]:.1f} % a {rango_deb[1]:.1f} %"
      f"  (2015: {tab.loc[2015, 'debutante_365'] if 2015 in tab.index else float('nan'):.1f} %)")

print("\n(D) Redundancia: primera_vez_365 frente a 'sin recurrencia del par en 12 meses'")
ok = d["pv_365"].notna() & d["n_par_12m"].notna()
red = 100 * (d.loc[ok, "pv_365"] == (d.loc[ok, "n_par_12m"] == 0)).mean()
red_cuota = 100 * (d.loc[ok & d["cuota_12m"].notna(), "pv_365"] ==
                   (d.loc[ok & d["cuota_12m"].notna(), "cuota_12m"] == 0)).mean()
print(f"  coincide con recurrencia = 0 en el {red:.2f} % y con cuota = 0 en el {red_cuota:.2f} %")
print("  Si es ~100 %, primera_vez_365 repite exactamente el cero de la recurrencia y sale del modelo.")

veredicto = "OK" if (c1 == 0 and c2 == 0 and cd >= 99 and cp >= 99) else "REVISAR"
print(f"\nVEREDICTO: {veredicto}" + ("  -> se puede correr el cuaderno 16" if veredicto == "OK"
                                   else "  -> pégame esta salida antes de correr el cuaderno 16"))

# 3) Registro de la enmienda (antes de cualquier evaluación de H1, H2 o H3)
enm = {
    "fecha": datetime.now().isoformat(timespec="seconds"),
    "id": "enmienda_2026-10-16",
    "motivo": ("Censura por la izquierda: la base empieza en 2015, así que en 2015 los indicadores con "
               "historial no tienen pasado y debutante/primera vez 'alguna vez' arrastran exceso hasta 2017. "
               "Piloto con 2023 (2024 intacto): la representación normal_sup supera al índice simple."),
    "decisiones": [
        "Se puntúan solo procesos con fecha_inicio_tender desde el 1-ene-2016; 2015 queda como historial.",
        "debutante = sin procesos del proveedor en los 365 días anteriores (ind_procesos_proveedor_12m == 0).",
        "primera vez con la entidad = sin procesos del par en los 365 días anteriores; al coincidir con "
        "recurrencia = 0, sale de los modelos por redundante.",
        "Representación de entrada de los modelos: normal_sup (rareza en el grupo a escala normal, mitad baja en 0).",
    ],
    "evidencia": {
        "piloto_2023_rapido": {"percentil": {"ensamble": 0.020, "indice": 0.027},
                               "normal_sup": {"ensamble": 0.085, "indice": 0.027, "dif": 0.058,
                                              "ic95": [0.024, 0.092], "inyecciones": 294}},
        "coherencia": {"debutante": c1, "primera_vez": c2},
        "coincidencia_recalculo_pct": {"debutante_365": round(cd, 3), "primera_vez_365": round(cp, 3)},
        "redundancia_primera_vez_pct": round(red, 3),
        "por_anio": tab.reset_index().to_dict(orient="records"),
    },
    "huellas": {"planA_indicadores": huella(IX), "planA_procesos_modelo": huella(PM)},
    "veredicto_verificacion": veredicto,
}
arch = os.path.join(ENMIENDAS, "enmienda_2026-10-16.json")
with open(arch, "w", encoding="utf-8") as f:
    json.dump(enm, f, ensure_ascii=False, indent=1, default=float)
print(f"\nEnmienda guardada en {arch} | huella {huella(arch)}")

In [ ]:
# =====================================================================================
# CELDA 1. Configuración: rutas, modo de corrida y parámetros fijados antes de modelar
# =====================================================================================
import os, time, json, math, hashlib, platform, warnings, unicodedata
from datetime import datetime
import numpy as np
import pandas as pd
import duckdb
import sklearn
from scipy.stats import spearmanr, norm
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.neural_network import MLPRegressor
from sklearn.exceptions import ConvergenceWarning

# Todo lo que se imprime con say() queda también en resumen_para_chat.txt (carpeta de la corrida)
SALIDA_TXT, CELDA_ACTUAL = {}, "celda 1"
def say(*args):
    texto = " ".join(str(a) for a in args)
    print(texto)
    SALIDA_TXT.setdefault(CELDA_ACTUAL, []).append(texto)

def guardar_txt():
    if "DIR_RUN" in globals():
        with open(os.path.join(DIR_RUN, "resumen_para_chat.txt"), "w", encoding="utf-8") as f:
            for celda, lineas in SALIDA_TXT.items():
                f.write(f"\n##### {celda.upper()} #####\n" + "\n".join(lineas) + "\n")

def empezar(nombre):
    """Marca el inicio de una celda: guarda lo anterior y limpia la salida de esta celda."""
    global CELDA_ACTUAL
    guardar_txt()
    CELDA_ACTUAL = nombre
    SALIDA_TXT[nombre] = []

# --- rutas (cámbialas solo si moviste la carpeta) ---
BASE   = r"C:\SERCOP_FINAL\plan_A"
SALIDA = r"C:\SERCOP_FINAL\modelos"

# --- modo de corrida ---
#   "COMPLETO" : todos los procesos puntuables (desde 2016). Es la corrida que vale para la tesis.
#   "RAPIDO"   : hasta 20.000 procesos por grupo, solo para probar que todo corre.
MODO = "COMPLETO"
MODO = MODO.upper()

# --- cómo ven los modelos cada variable (ver la explicación en el documento) ---
#   "normal_sup" : rareza dentro del grupo pasada a escala normal, contando solo la cola alta (riesgo).
#                  Elegida por el grupo tras el piloto con 2023 (enmienda del 16-oct-2026).
#   "percentil"  : tal como sale de la base; queda solo para comparar.
REPRESENTACION = "normal_sup"

SEMILLA = 20261001                      # semilla global del proyecto

# --- parámetros fijados en el anteproyecto (NO se cambian mirando resultados) ---
IF_ARBOLES    = 100                     # Isolation Forest: árboles (Liu et al., 2008)
IF_SUBMUESTRA = 256                     # Isolation Forest: tamaño de submuestra por árbol
LOF_VECINOS   = 20                      # Local Outlier Factor: vecinos (Breunig et al., 2000)
CORTE_TOP     = 0.01                    # lista priorizada: 1 % superior de cada grupo
MIN_PATRON    = 1000                    # patrón de disponibilidad con modelo propio

# --- enmienda del 16-oct-2026 (decidida antes de evaluar H1, H2 y H3) ---
ANIO_INICIO   = 2016                    # 2015 solo sirve de historial: no se puntúa

# --- parámetros de cómputo (no cambian la definición del modelo, solo el costo) ---
MIN_GRUPO        = 100                  # grupos más chicos se reportan pero no se modelan
REF_MAX          = 50_000               # muestra de referencia máxima por modelo
AE_MAX_ENTRENO   = 50_000               # filas máximas para entrenar el autoencoder
HUERFANO_MAX     = 10_000               # lo mismo, para los modelos de patrones pequeños
RAPIDO_MAX_GRUPO = 20_000               # procesos por grupo en modo RAPIDO

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M") + "_" + MODO.lower() + "_" + REPRESENTACION
DIR_RUN = os.path.join(SALIDA, RUN_ID)
os.makedirs(DIR_RUN, exist_ok=True)
T_INICIO = time.time()
ALERTAS = []                            # aquí se acumulan los avisos que se resumen en la celda 9

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")
warnings.filterwarnings("ignore", category=RuntimeWarning, message="Mean of empty slice")
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)

say("Corrida:", RUN_ID)
say("Carpeta de salida:", DIR_RUN)
say(f"Python {platform.python_version()} | numpy {np.__version__} | pandas {pd.__version__} | "
      f"scikit-learn {sklearn.__version__} | duckdb {duckdb.__version__} | núcleos {os.cpu_count()}")
say(f"Parámetros fijados: IF {IF_ARBOLES} árboles x {IF_SUBMUESTRA} | LOF k={LOF_VECINOS} | "
      f"corte {CORTE_TOP:.0%} | patrón mínimo {MIN_PATRON} | representación '{REPRESENTACION}' | "
      f"se puntúa desde {ANIO_INICIO}")
if REPRESENTACION not in ("percentil", "normal_sup"):
    raise SystemExit("REPRESENTACION debe ser 'percentil' o 'normal_sup'.")

In [ ]:
# =====================================================================================
# CELDA 2. Carga de la base entregada y armado de los grupos comparables
#          (NO se carga planA_validacion_sri: la marca del SRI se usa una sola vez, al final)
# =====================================================================================
empezar("celda 2")
def ruta(nombre):
    return os.path.join(BASE, nombre).replace("\\", "/")

def huella(path, n=16):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()[:n]

def a_float(s):
    """Convierte booleanos, enteros o nulos a float (nulo -> NaN) sin errores de tipo."""
    try:
        return pd.Series(pd.array(s, dtype="Float64").to_numpy(dtype="float64", na_value=np.nan), index=s.index)
    except Exception:
        return pd.to_numeric(s.astype("object"), errors="coerce").astype("float64")

def guardar_parquet(tabla, nombre):
    """Guarda en la carpeta de la corrida; si pandas no tiene motor de parquet, usa DuckDB."""
    destino = os.path.join(DIR_RUN, nombre)
    try:
        tabla.to_parquet(destino)
    except ImportError:
        c = duckdb.connect()
        c.register("t_guardar", tabla.reset_index())
        c.execute(f"COPY t_guardar TO '{destino.replace(chr(92), '/')}' (FORMAT PARQUET)")
        c.close()

def sin_tildes(texto):
    t = unicodedata.normalize("NFKD", str(texto))
    return "".join(c for c in t if not unicodedata.combining(c)).lower().strip()

t_celda = time.time()
IX = ruta("planA_indicadores.parquet")
PM = ruta("planA_procesos_modelo.parquet")
for p in (IX, PM):
    if not os.path.exists(p):
        raise SystemExit(f"No encuentro {p}. Revisa BASE en la celda 1.")
HUELLA_IX, HUELLA_PM = huella(IX), huella(PM)

con = duckdb.connect()
ix = con.execute(f"SELECT * FROM read_parquet('{IX}')").df()
cols_pm = [r[0] for r in con.execute(f"DESCRIBE SELECT * FROM read_parquet('{PM}')").fetchall()]
pedir = ["ocid"] + [c for c in ["familia", "fecha_inicio_tender", "monto_final", "entidad_ruc", "proveedor_ruc_final"]
                    if c in cols_pm and c not in ix.columns]
pm = con.execute(f"SELECT {', '.join(pedir)} FROM read_parquet('{PM}')").df()
say(f"planA_indicadores: {ix.shape[0]:,} filas x {ix.shape[1]} columnas | huella {HUELLA_IX}")
say(f"planA_procesos_modelo: {pm.shape[0]:,} filas; se toman {pedir[1:]} | huella {HUELLA_PM}")
say("\nColumnas de planA_indicadores:")
say(", ".join(ix.columns))

df = ix.merge(pm, on="ocid", how="left")
if len(df) != len(ix) or df["ocid"].duplicated().any():
    raise SystemExit("La unión por ocid cambió el número de filas: revisar antes de seguir.")
say(f"\nProcesos de indicadores sin pareja en procesos_modelo: {(~ix['ocid'].isin(pm['ocid'])).sum():,}")
if "familia" not in df.columns:
    raise SystemExit("No encuentro la columna 'familia' en ninguno de los dos archivos.")

# fecha del proceso = inicio del período de ofertas
col_fecha = next((c for c in ["fecha_inicio_tender", "fecha_proceso"] if c in df.columns), None)
if col_fecha is None:
    raise SystemExit("No encuentro la fecha del proceso (fecha_inicio_tender).")
fecha = pd.to_datetime(df[col_fecha], errors="coerce")
try:
    if fecha.dt.tz is not None:
        fecha = fecha.dt.tz_convert("America/Guayaquil").dt.tz_localize(None)
except AttributeError:
    pass
say(f"Fecha del proceso tomada de '{col_fecha}'; procesos sin fecha: {int(fecha.isna().sum()):,}")
df["anio_proc"] = fecha.dt.year
df["periodo_calc"] = np.where(fecha >= pd.Timestamp("2025-10-07"), "post", "pre")
if "periodo" in df.columns:
    coinc = (df["periodo"].astype(str).str.lower().str.startswith("post") == (df["periodo_calc"] == "post")).mean()
    say(f"Coincidencia entre 'periodo' de la base y el corte por fecha: {coinc:.4%}")
df["grupo"] = df["familia"].astype(str) + " | " + df["periodo_calc"]
if "monto_final" not in df.columns:
    df["monto_final"] = np.nan
df["monto_final"] = a_float(df["monto_final"])

# Control de calibración sobre la base completa (antes de quitar 2015 y de muestrear):
# cada percentil de la base debe tener media 0,500 dentro de su grupo
say("\nControl de calibración de la base (media de cada percentil dentro de su grupo; debe dar 0,500):")
filas = []
for c in ["pct_precio_alto", "pct_monto_capital", "pct_proveedor_nuevo", "pct_plazo_corto", "pct_firma_rapida",
          "pct_fraccionamiento_pos", "pct_cuota_pos", "pct_recurrencia_pos"]:
    if c in df.columns:
        s_ = a_float(df[c])
        g_ = s_.groupby(df["grupo"])
        m = g_.mean()[g_.count() >= 200]                 # solo grupos con al menos 200 valores
        filas.append(dict(columna=c, media_min=m.min(), media_max=m.max(),
                          grupo_mas_lejos=(m - 0.5).abs().idxmax() if m.notna().any() else "-"))
cal = pd.DataFrame(filas)
say(cal.round(4).to_string(index=False))
lejos = cal[(cal["media_min"] < 0.495) | (cal["media_max"] > 0.505)]
if len(lejos):
    ALERTAS.append("Percentiles de la base con media lejos de 0,5 en algún grupo: " + ", ".join(lejos["columna"]))

# Debutante con ventana de 12 meses (enmienda): sin procesos del proveedor en los 365 días anteriores.
# Sale de la misma columna de historial que usan cuota y recurrencia (verificada en la celda 0).
if "ind_procesos_proveedor_12m" not in df.columns:
    raise SystemExit("No encuentro ind_procesos_proveedor_12m: corre primero la celda 0 y pégame su salida.")
n12 = a_float(df["ind_procesos_proveedor_12m"])
df["ind_proveedor_debutante_365"] = np.where(n12.isna(), np.nan, (n12 == 0).astype(float))

# 2015 queda solo como historial (enmienda): se retira de lo que se puntúa
quitados = df.loc[df["anio_proc"] < ANIO_INICIO].groupby("grupo").size()
df = df.loc[df["anio_proc"] >= ANIO_INICIO].reset_index(drop=True)
say(f"\nProcesos anteriores a {ANIO_INICIO} retirados (siguen sirviendo de historial): {int(quitados.sum()):,}")
say(f"Procesos puntuables: {len(df):,}")

TAM_REAL = df.groupby("grupo").size()            # tamaño de cada grupo puntuable en la base completa
if MODO == "RAPIDO":
    df = (df.sample(frac=1.0, random_state=SEMILLA)
            .groupby("grupo", sort=False).head(RAPIDO_MAX_GRUPO)
            .sort_values("ocid").reset_index(drop=True))
    say(f"\nModo RAPIDO: se trabaja con {len(df):,} procesos (hasta {RAPIDO_MAX_GRUPO:,} por grupo).")

tab = (df.groupby("grupo").agg(procesos=("ocid", "size"),
                               de_2023=("anio_proc", lambda s: int((s == 2023).sum())),
                               anio_min=("anio_proc", "min"), anio_max=("anio_proc", "max"),
                               millones=("monto_final", lambda s: round(s.sum() / 1e6, 1))))
tab.insert(0, "en_la_base", TAM_REAL)
tab.insert(1, f"quitados_antes_{ANIO_INICIO}", quitados.reindex(tab.index).fillna(0).astype(int))
tab = tab.sort_values("en_la_base", ascending=False)
say("\nGrupos comparables (familia | período):")
say(tab.to_string())
say("\nRecordatorio: planA_validacion_sri NO se cargó. Ninguna cifra de esta corrida usa la marca del SRI.")
say(f"[celda 2: {time.time() - t_celda:.0f} s]")

In [ ]:
# =====================================================================================
# CELDA 3. Codificación de las variables del modelo (sin imputar constantes)
#   rango      -> percentil del grupo (ya viene en la base); si falta el dato queda vacío (NaN)
#   rango_pos  -> percentil entre los procesos donde el fenómeno ocurre; 0 cuando no ocurre
#                 (cero real: no hubo historial o no hubo recompra), nunca un relleno
#   binaria    -> 0/1; vacío si falta
#   dif        -> diferencia con la mediana de oferentes del grupo, llevada a 0-1 dentro del grupo;
#                 1 cuando hubo un solo oferente (el extremo real de la escala)
# =====================================================================================
empezar("celda 3")
t_celda = time.time()

def pct_mitad_salto(s, grupos):
    """Percentil a mitad de salto dentro de cada grupo (los vacíos quedan vacíos)."""
    r = s.groupby(grupos).rank(method="average")
    n = s.notna().groupby(grupos).transform("sum")
    return (r - 0.5) / n

VARIABLES = {
    # nombre             columna en la base                  tipo         entra a la línea base
    "precio_alto":       ("pct_precio_alto",                 "rango",     True),
    "monto_capital":     ("pct_monto_capital",               "rango",     True),
    "proveedor_nuevo":   ("pct_proveedor_nuevo",             "rango",     True),
    "plazo_corto":       ("pct_plazo_corto",                 "rango",     True),
    "firma_rapida":      ("pct_firma_rapida",                "rango",     True),
    "recompra_30d":      ("pct_fraccionamiento_pos",         "rango_pos", True),
    "cuota_entidad":     ("pct_cuota_pos",                   "rango_pos", True),
    "recurrencia_par":   ("pct_recurrencia_pos",             "rango_pos", True),
    "oferente_unico":    ("ind_oferente_unico",              "binaria",   True),
    "poca_concurrencia": ("dif_concurrencia_grupo",          "dif",       True),
    # primera_vez_ent sale de los modelos: con la ventana de 12 meses coincide exactamente con
    # "recurrencia = 0", así que no aporta información nueva (enmienda del 16-oct-2026)
    "debutante":         ("ind_proveedor_debutante_365",     "binaria",   True),
}
# columnas de origen de las variables en dos partes: la que dice si el fenómeno ocurre (> 0) y la
# que mide su tamaño (solo se usa si hubiera que reconstruir el percentil)
ORIGEN_POS = {"recompra_30d":    ["ind_fraccionamiento_marca", "ind_fraccionamiento_monto_30d"],
              "cuota_entidad":   ["ind_cuota_prov_en_entidad_12m"],
              "recurrencia_par": ["ind_recurrencia_par_12m"]}
MAGNITUD_POS = {"recompra_30d":    ["ind_fraccionamiento_monto_30d"],
                "cuota_entidad":   ["ind_cuota_prov_en_entidad_12m"],
                "recurrencia_par": ["ind_recurrencia_par_12m"]}

def primera(cols):
    return next((c for c in cols if c in df.columns), None)

# Si una columna no existe con el nombre esperado, se reconstruye desde su origen (y se avisa)
RECONSTRUIDAS = []
for v, origenes in MAGNITUD_POS.items():
    col, origen = VARIABLES[v][0], primera(origenes)
    if col not in df.columns and origen is not None:
        mag = a_float(df[origen])
        df[col] = pct_mitad_salto(mag.where(mag > 0), df["grupo"])
        RECONSTRUIDAS.append(f"{v} desde {origen}")
col_nof = primera(["ind_n_oferentes", "n_oferentes"])
if col_nof is not None:
    nof = a_float(df[col_nof])
    if "ind_oferente_unico" not in df.columns:
        df["ind_oferente_unico"] = np.where(nof.isna(), np.nan, (nof == 1).astype(float))
        RECONSTRUIDAS.append(f"oferente_unico desde {col_nof}")
    if "dif_concurrencia_grupo" not in df.columns:
        df["dif_concurrencia_grupo"] = (nof.groupby(df["grupo"]).transform("median") - nof).where(nof >= 2)
        RECONSTRUIDAS.append(f"poca_concurrencia desde {col_nof}")

X = pd.DataFrame(index=df.index)
say("Variables del modelo y columna de origen:")
for v, (col, tipo, _) in list(VARIABLES.items()):
    if col not in df.columns:
        say(f"  {v:<18} NO ENCONTRADA ({col}) -> queda fuera")
        ALERTAS.append(f"Variable {v} no encontrada en la base (columna {col}).")
        VARIABLES.pop(v)
        continue
    s = a_float(df[col])
    nota = ""
    if tipo == "rango_pos":
        origen = primera(ORIGEN_POS[v])
        if origen is not None:
            o = a_float(df[origen])
            ocurre, no_ocurre = o > 0, o <= 0
            mal = int((s.isna() & ocurre).sum() + ((s > 0) & no_ocurre).sum())
            nota = f"control de ceros con {origen}: {mal:,} filas incoherentes" + ("  <- REVISAR" if mal else " (ok)")
            if mal:
                ALERTAS.append(f"{v}: {mal:,} filas donde el percentil y su origen no coinciden.")
            s = s.where(~(no_ocurre & s.isna()), 0.0)
        else:
            nota = "sin columna de origen: vacío se toma como 'no ocurre'"
            s = s.fillna(0.0)
    X[v] = s
    say(f"  {v:<18} {col:<32} {tipo:<10} cobertura {100 * s.notna().mean():5.1f} %   {nota}")
if RECONSTRUIDAS:
    say("Reconstruidas porque la columna esperada no existe:", "; ".join(RECONSTRUIDAS))
    ALERTAS.append("Variables reconstruidas desde su origen: " + "; ".join(RECONSTRUIDAS))
if X.shape[1] < 6:
    raise SystemExit("Faltan demasiadas variables. Pega en el chat la lista de columnas de la celda 2.")

# poca_concurrencia: escala 0-1 dentro de cada grupo, 1 = un solo oferente
if "poca_concurrencia" in X.columns:
    pc = X["poca_concurrencia"].copy()
    for g, idx in df.groupby("grupo").groups.items():
        vals = pc.loc[idx].dropna()
        if len(vals) >= 30:
            lo, hi = np.nanpercentile(vals, 1), np.nanpercentile(vals, 99)
            if hi > lo:
                pc.loc[idx] = ((pc.loc[idx].clip(lo, hi) - lo) / (hi - lo))
            else:
                pc.loc[idx] = np.where(pc.loc[idx].notna(), 0.5, np.nan)
        else:
            pc.loc[idx] = np.where(pc.loc[idx].notna(), 0.5, np.nan)
    if "oferente_unico" in X.columns:
        pc = pc.where(~(X["oferente_unico"] == 1), 1.0)
        pc = pc.where(~X["oferente_unico"].isna(), np.nan)
    X["poca_concurrencia"] = pc

ORIENTADAS = [v for v, (_, _, base) in VARIABLES.items() if base]
TIPO = {v: t for v, (_, t, _) in VARIABLES.items()}
CORTO = {"precio_alto": "precio", "monto_capital": "capital", "proveedor_nuevo": "prov_nuevo",
         "plazo_corto": "plazo", "firma_rapida": "firma", "recompra_30d": "recompra",
         "cuota_entidad": "cuota", "recurrencia_par": "recurr", "oferente_unico": "of_unico",
         "poca_concurrencia": "poca_conc", "primera_vez_ent": "primera_vez", "debutante": "debutante"}

# control de orientación (alto = anómalo); la calibración ya se revisó en la celda 2
def control_orientacion(var, origen, signo):
    if var in X.columns and origen in df.columns:
        a, b = X[var], a_float(df[origen])
        ok = a.notna() & b.notna() & ((a > 0) if TIPO[var] == "rango_pos" else True)
        if ok.sum() > 100:
            r = spearmanr(a[ok], b[ok]).correlation
            estado = "ok" if np.sign(r) == signo else "REVISAR ORIENTACIÓN"
            if estado != "ok":
                ALERTAS.append(f"Orientación dudosa: {var} frente a {origen} (rho {r:+.2f}).")
            say(f"  {var:<18} vs {origen:<32} rho = {r:+.3f}  (esperado {'+' if signo > 0 else '-'})  {estado}")
say("\nControl de orientación (alto = más anómalo):")
control_orientacion("precio_alto", "ind_desv_precio_unit_z", +1)
control_orientacion("monto_capital", "ind_monto_sobre_capital", +1)
control_orientacion("proveedor_nuevo", "ind_antiguedad_dias", -1)
control_orientacion("plazo_corto", "ind_plazo_ofertas_dias", -1)
control_orientacion("firma_rapida", "ind_plazo_adj_firma_dias", -1)
control_orientacion("recompra_30d", "ind_fraccionamiento_monto_30d", +1)
control_orientacion("cuota_entidad", "ind_cuota_prov_en_entidad_12m", +1)
control_orientacion("recurrencia_par", "ind_recurrencia_par_12m", +1)
if col_nof is not None:
    control_orientacion("poca_concurrencia", col_nof, -1)

cob = (X.notna().groupby(df["grupo"]).mean() * 100).round(1)
cob.insert(0, "procesos", df.groupby("grupo").size())
say("\nCobertura de cada variable por grupo (% de procesos con dato):")
say(cob.sort_values("procesos", ascending=False).rename(columns=CORTO).to_string())

guardar_parquet(X, "X_codificado.parquet")
guardar_parquet(df[["ocid", "grupo", "familia", "periodo_calc", "anio_proc", "monto_final"]
                   + [c for c in ["entidad_ruc", "proveedor_ruc_final"] if c in df.columns]], "llaves.parquet")
say(f"[celda 3: {time.time() - t_celda:.0f} s]")

In [ ]:
# =====================================================================================
# CELDA 4. Patrones de disponibilidad: qué variables usa cada modelo en cada grupo
#   - Se retiran del grupo las variables casi ausentes (<1 %) o constantes (>= 99,5 % un valor).
#   - Cada patrón con >= MIN_PATRON procesos tiene modelo propio ("grande").
#   - Un patrón menor se puntúa con el modelo grande que usa solo variables que él sí tiene;
#     si no hay ninguno, recibe un modelo propio ajustado sin la variable que le falta ("pequeño").
# =====================================================================================
empezar("celda 4")
def variables_del_grupo(Xg):
    usar, fuera = [], []
    for v in Xg.columns:
        s = Xg[v].dropna()
        if len(s) < max(30, 0.01 * len(Xg)):
            fuera.append(f"{v} (sin dato)")
        elif s.value_counts(normalize=True).iloc[0] >= 0.995:
            fuera.append(f"{v} (constante)")
        else:
            usar.append(v)
    return usar, fuera

def plan_de_modelos(Xg, usar):
    disp = Xg[usar].notna().to_numpy()
    codigo = (disp * (1 << np.arange(len(usar)))).sum(axis=1)
    texto = {c: "".join("1" if (c >> j) & 1 else "0" for j in range(len(usar))) for c in np.unique(codigo)}
    clave = pd.Series([texto[c] for c in codigo], index=Xg.index)
    conteo = clave.value_counts()
    grandes = [k for k, n in conteo.items() if n >= MIN_PATRON] or [conteo.index[0]]
    def vars_de(k): return [u for u, b in zip(usar, k) if b == "1"]
    asign = {}
    for k in conteo.index:
        if k in grandes:
            asign[k] = k
            continue
        mias = set(vars_de(k))
        cand = [g for g in grandes if set(vars_de(g)) <= mias]
        asign[k] = max(cand, key=lambda g: (len(vars_de(g)), conteo[g])) if cand else k
    modelos = sorted(set(asign.values()), key=lambda k: -conteo.get(k, 0))
    return clave, conteo, asign, {m: vars_de(m) for m in modelos}, set(grandes)

PLAN, NO_MODELADOS, filas = {}, {}, []
for g in df.groupby("grupo").size().sort_values(ascending=False).index:
    idx = df.index[df["grupo"] == g]
    Xg = X.loc[idx]
    usar, fuera = variables_del_grupo(Xg)
    if len(idx) < MIN_GRUPO or len(usar) < 2:
        NO_MODELADOS[g] = f"{len(idx)} procesos, {len(usar)} variables útiles"
        continue
    clave, conteo, asign, vars_modelo, grandes = plan_de_modelos(Xg, usar)
    PLAN[g] = dict(idx=idx, usar=usar, clave=clave, asign=asign, vars_modelo=vars_modelo, grandes=grandes)
    modelo_de = clave.map(asign)
    filas.append(dict(grupo=g, procesos=len(idx), variables=len(usar), patrones=len(conteo),
                      modelos_grandes=sum(m in grandes for m in vars_modelo),
                      modelos_pequenos=sum(m not in grandes for m in vars_modelo),
                      pct_en_modelo_principal=round(100 * (modelo_de == conteo.index[0]).mean(), 1),
                      pct_en_modelos_pequenos=round(100 * (~modelo_de.isin(grandes)).mean(), 2),
                      vars_modelo_principal=conteo.index[0].count("1"),
                      fuera=", ".join(fuera)))
say(pd.DataFrame(filas).to_string(index=False))
if NO_MODELADOS:
    say("\nGrupos que no se modelan:", NO_MODELADOS)

In [ ]:
# =====================================================================================
# CELDA 5. Ajuste de Isolation Forest, Local Outlier Factor y autoencoder por grupo y patrón,
#          índice simple (línea base), lista del 1 % superior, núcleo de consenso y explicación
# =====================================================================================
empezar("celda 5")
def pct_vs_ref(ref, x):
    """Percentil a mitad de salto de cada valor de x frente a la distribución de referencia ref."""
    ref = np.sort(np.asarray(ref, dtype=float))
    lo = np.searchsorted(ref, x, side="left")
    hi = np.searchsorted(ref, x, side="right")
    return (lo + 0.5 * (hi - lo)) / len(ref)

def ajustar_un_modelo(Xtr, Xsc, semilla, ref_max, ae_max):
    """Entrena los tres detectores con Xtr (procesos que tienen todas las variables del modelo)
    y puntúa Xsc. Devuelve percentiles (alto = más anómalo) frente a una referencia tomada de Xtr."""
    rng = np.random.default_rng(semilla)
    n, d = Xtr.shape
    R = Xtr[rng.choice(n, size=min(n, ref_max), replace=False)]
    out, tiempos = {}, {}
    # Isolation Forest
    t = time.time()
    iso = IsolationForest(n_estimators=IF_ARBOLES, max_samples=min(IF_SUBMUESTRA, n),
                          random_state=semilla, n_jobs=-1).fit(Xtr)
    out["if"] = pct_vs_ref(-iso.score_samples(R), -iso.score_samples(Xsc))
    tiempos["if"] = time.time() - t
    # Local Outlier Factor (modo novedad). Se ajusta sobre las filas distintas de la referencia.
    # Un proceso idéntico a una fila de la referencia recibe el LOF de esa fila, calculado sin
    # contarse a sí misma; así todos los procesos se miden con la misma regla.
    t = time.time()
    U, inv = np.unique(R, axis=0, return_inverse=True)
    inv = np.asarray(inv).ravel()
    if len(U) > LOF_VECINOS + 5:
        lof = LocalOutlierFactor(n_neighbors=LOF_VECINOS, novelty=True, n_jobs=-1).fit(U)
        lof_u = -lof.negative_outlier_factor_
        _, inv_all = np.unique(np.vstack([U, Xsc]), axis=0, return_inverse=True)
        inv_all = np.asarray(inv_all).ravel()
        pos = np.full(inv_all.max() + 1, -1)
        pos[inv_all[:len(U)]] = np.arange(len(U))
        m = pos[inv_all[len(U):]]
        s = np.empty(len(Xsc))
        s[m >= 0] = lof_u[m[m >= 0]]
        if (m < 0).any():
            s[m < 0] = -lof.score_samples(Xsc[m < 0])
        out["lof"] = pct_vs_ref(lof_u[inv], s)
    else:
        out["lof"] = np.full(len(Xsc), np.nan)
    tiempos["lof"] = time.time() - t
    # Autoencoder: red simétrica con capa intermedia de la mitad de las variables
    t = time.time()
    if d >= 3:
        tr_i = rng.choice(n, size=min(n, ae_max), replace=False)
        ae = MLPRegressor(hidden_layer_sizes=(max(2, d // 2),), activation="relu", solver="adam",
                          alpha=1e-4, batch_size=256, learning_rate_init=1e-3, max_iter=200,
                          early_stopping=True, validation_fraction=0.1, n_iter_no_change=10,
                          random_state=semilla)
        ae.fit(Xtr[tr_i], Xtr[tr_i])
        err_ref = ((ae.predict(R) - R) ** 2).mean(axis=1)
        err_sc = ((ae.predict(Xsc) - Xsc) ** 2).mean(axis=1)
        out["ae"] = pct_vs_ref(err_ref, err_sc)
    else:
        out["ae"] = np.full(len(Xsc), np.nan)
    tiempos["ae"] = time.time() - t
    return out, tiempos

def top_k(valores, frac, semilla):
    """Marca el frac superior (exactamente k procesos con puntaje; empates al azar con semilla fija)."""
    v = np.asarray(valores, dtype=float)
    fin = np.isfinite(v)
    marca = np.zeros(len(v), dtype=bool)
    if fin.sum() == 0:
        return marca
    k = max(1, int(math.ceil(frac * fin.sum())))
    desempate = np.random.default_rng(semilla).random(len(v))
    orden = np.lexsort((desempate, -np.where(fin, v, -np.inf)))
    marca[orden[:k]] = True
    return marca

def rareza(Xg_df):
    """Percentil a mitad de salto de cada variable dentro del conjunto (alto = valor más alto)."""
    return (Xg_df.rank(method="average") - 0.5) / Xg_df.notna().sum()

def entrada_modelo(Xg_df):
    """Lo que reciben los modelos. 'percentil': los valores tal cual. 'normal_sup': la rareza de cada
    valor dentro del grupo en escala normal, con la mitad baja (sin riesgo) llevada a 0."""
    if REPRESENTACION == "normal_sup":
        z = norm.ppf(rareza(Xg_df).clip(1e-9, 1 - 1e-9).to_numpy(dtype=float))
        return pd.DataFrame(np.maximum(z, 0.0), index=Xg_df.index, columns=Xg_df.columns).where(Xg_df.notna())
    return Xg_df

def puntuar_grupo(plan, Xg_df, semilla, verbose=False):
    """Ajusta todos los modelos de un grupo. Devuelve percentiles por proceso, tiempos y rareza."""
    res = pd.DataFrame(index=Xg_df.index)
    Xm = entrada_modelo(Xg_df[plan["usar"]])
    for m in ["if", "lof", "ae"]:
        res[m] = np.nan
    res["modelo"] = ""
    t_mod = {"if": 0.0, "lof": 0.0, "ae": 0.0}
    clave = plan["clave"].loc[Xg_df.index]
    modelo_de = clave.map(plan["asign"])
    for j, (mk, vs) in enumerate(plan["vars_modelo"].items(), start=1):
        tm = time.time()
        destino = modelo_de.index[modelo_de == mk]
        if len(destino) == 0 or len(vs) == 0:
            continue
        tr_mask = Xg_df[vs].notna().all(axis=1)
        if tr_mask.sum() < 30:
            continue
        tope = (REF_MAX, AE_MAX_ENTRENO) if mk in plan["grandes"] else (HUERFANO_MAX, HUERFANO_MAX)
        out, tt = ajustar_un_modelo(Xm.loc[tr_mask, vs].to_numpy(dtype=float),
                                    Xm.loc[destino, vs].to_numpy(dtype=float), semilla, *tope)
        for m in out:
            res.loc[destino, m] = out[m]
            t_mod[m] += tt[m]
        res.loc[destino, "modelo"] = mk
        if verbose:
            say(f"      modelo {j}/{len(plan['vars_modelo'])}: {len(vs)} variables, entrena con "
                  f"{int(tr_mask.sum()):,}, puntúa {len(destino):,}  ({time.time() - tm:.0f} s)")
    res["ensamble"] = res[["if", "lof", "ae"]].mean(axis=1, skipna=True)
    rz = rareza(Xg_df[plan["usar"]])
    orient = [v for v in ORIENTADAS if v in plan["usar"]]
    res["base"] = rz[orient].mean(axis=1, skipna=True) if orient else np.nan
    res["n_vars_disp"] = Xg_df[plan["usar"]].notna().sum(axis=1)
    return res, t_mod, rz

def explicar(rz, cols, n=3):
    """Para cada fila, las n variables con valor más alto dentro del grupo."""
    a = rz[cols].to_numpy(dtype=float)
    a = np.where(np.isfinite(a), a, -1.0)
    orden = np.argsort(-a, axis=1)[:, :n]
    return [" | ".join(f"{cols[j]} {a[i, j]:.2f}" for j in orden[i] if a[i, j] >= 0) for i in range(len(a))]

def marcar_tops(res, semilla):
    for m in ["ensamble", "if", "lof", "ae", "base"]:
        res[f"top_{m}"] = top_k(res[m], CORTE_TOP, semilla)
    res["consenso"] = res[["top_if", "top_lof", "top_ae"]].sum(axis=1) >= 2
    return res

t0 = time.time()
partes, tiempos = [], []
for g, plan in PLAN.items():
    tg = time.time()
    res_g, t_mod, rz = puntuar_grupo(plan, X.loc[plan["idx"]], SEMILLA, verbose=len(plan["idx"]) >= 50_000)
    res_g = marcar_tops(res_g, SEMILLA)
    orient = [v for v in ORIENTADAS if v in plan["usar"]]
    res_g["explica"] = ""
    sel = res_g.index[res_g["top_ensamble"]]
    res_g.loc[sel, "explica"] = explicar(rz.loc[sel], orient)
    res_g["grupo"] = g
    partes.append(res_g)
    tiempos.append(dict(grupo=g, procesos=len(res_g), modelos=len(plan["vars_modelo"]),
                        seg_if=round(t_mod["if"], 1), seg_lof=round(t_mod["lof"], 1),
                        seg_ae=round(t_mod["ae"], 1), seg_total=round(time.time() - tg, 1)))
    say(f"  {g:<42} {len(res_g):>8,} procesos  {time.time() - tg:7.1f} s")
RES = pd.concat(partes).sort_index()
RES = RES.join(df[["ocid", "familia", "periodo_calc", "anio_proc", "monto_final"]
                  + [c for c in ["entidad_ruc", "proveedor_ruc_final"] if c in df.columns]])
guardar_parquet(RES, "puntajes.parquet")
say(f"\nTiempo total de ajuste: {(time.time() - t0) / 60:.1f} min")
say(pd.DataFrame(tiempos).to_string(index=False))
say(f"\nLista priorizada: {int(RES['top_ensamble'].sum()):,} procesos (1 % de cada grupo); "
      f"núcleo de consenso dentro de ella: {int((RES['top_ensamble'] & RES['consenso']).sum()):,}")

In [ ]:
# =====================================================================================
# CELDA 6. Diagnósticos sin etiquetas: concordancia, qué señales tiene el 1 % superior,
#          monto, años, concentración y modelos pequeños. No se imprime ningún RUC.
# =====================================================================================
empezar("celda 6")
t_celda = time.time()

def jaccard(a, b):
    a, b = np.asarray(a, bool), np.asarray(b, bool)
    u = (a | b).sum()
    return (a & b).sum() / u if u else np.nan

def rho(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    ok = np.isfinite(a) & np.isfinite(b)
    if ok.sum() <= 10 or np.std(a[ok]) == 0 or np.std(b[ok]) == 0:
        return np.nan
    return spearmanr(a[ok], b[ok]).correlation

# (6a) Concordancia
conc = []
for g, r in RES.groupby("grupo"):
    a, b, c, e, bs = (r[m].to_numpy(float) for m in ["if", "lof", "ae", "ensamble", "base"])
    rs = [rho(a, b), rho(a, c), rho(b, c)]
    rbar = np.nanmean(rs) if np.isfinite(rs).any() else np.nan
    conc.append(dict(grupo=g, procesos=len(r), top=int(r.top_ensamble.sum()),
                     rho_if_lof=rs[0], rho_if_ae=rs[1], rho_lof_ae=rs[2],
                     W_kendall=(2 * rbar + 1) / 3 if np.isfinite(rbar) else np.nan,
                     jac_if_lof=jaccard(r.top_if, r.top_lof), jac_if_ae=jaccard(r.top_if, r.top_ae),
                     jac_lof_ae=jaccard(r.top_lof, r.top_ae),
                     consenso_pct=100 * (r.top_ensamble & r.consenso).sum() / max(1, r.top_ensamble.sum()),
                     rho_ens_base=rho(e, bs), jac_ens_base=jaccard(r.top_ensamble, r.top_base),
                     rho_ens_nvars=rho(e, r.n_vars_disp.to_numpy(float)),
                     rho_ens_monto=rho(e, np.log1p(r.monto_final.to_numpy(float)))))
CONC = pd.DataFrame(conc).sort_values("procesos", ascending=False)
say("(6a) Concordancia entre modelos y con el índice simple, por grupo")
say(CONC.round(3).to_string(index=False))
say("Lectura: W >= 0,5 = acuerdo suficiente; jac = parte compartida del 1 % superior (1 = misma lista);"
      " |rho_ens_nvars| > 0,1 = el puntaje depende de cuántas variables tiene el proceso (regla del"
      " anteproyecto); rho_ens_monto alto = se señala por tamaño.")
for g in CONC.loc[CONC["rho_ens_nvars"].abs() > 0.1, "grupo"]:
    ALERTAS.append(f"{g}: falla el control de cobertura (|rho puntaje vs n.º de variables| > 0,1).")
for g in CONC.loc[CONC["W_kendall"] < 0.5, "grupo"]:
    ALERTAS.append(f"{g}: W < 0,5; por regla del anteproyecto su lista exige acuerdo de 2 de 3 modelos.")
# Regla del anteproyecto aplicada a la lista: en los grupos con W < 0,5 solo queda el núcleo de consenso
RES["priorizado"] = RES["top_ensamble"]
for g in CONC.loc[CONC["W_kendall"] < 0.5, "grupo"]:
    m = RES["grupo"] == g
    RES.loc[m, "priorizado"] = RES.loc[m, "top_ensamble"] & RES.loc[m, "consenso"]
guardar_parquet(RES, "puntajes.parquet")
say(f"Lista priorizada final (con la regla de W aplicada): {int(RES['priorizado'].sum()):,} procesos")
for g in CONC.loc[CONC["rho_ens_monto"].abs() > 0.3, "grupo"]:
    ALERTAS.append(f"{g}: el puntaje sigue al monto (|rho| > 0,3).")

# (6b) Qué señales tiene el 1 % superior frente al resto
def activa(v, s):
    if TIPO[v] == "rango_pos":
        return s > 0                    # el fenómeno ocurre
    if TIPO[v] == "binaria":
        return s == 1
    return s >= 0.9                     # rango y dif: en el 10 % más alto del grupo

Xr = X.loc[RES.index]
lift, filas_pool = [], {}
for g, r in RES.groupby("grupo"):
    t = r["top_ensamble"].to_numpy(bool)
    for v in PLAN[g]["usar"]:
        s = Xr.loc[r.index, v]
        dato = s.notna().to_numpy()
        act = activa(v, s).to_numpy() & dato
        pt = 100 * act[t].sum() / max(1, dato[t].sum())
        pr = 100 * act[~t].sum() / max(1, dato[~t].sum())
        lift.append(dict(grupo=g, variable=v, pct_top=pt, pct_resto=pr,
                         razon=pt / pr if pr > 0 else np.nan))
        acc = filas_pool.setdefault(v, [0, 0, 0, 0])
        acc[0] += act[t].sum(); acc[1] += dato[t].sum(); acc[2] += act[~t].sum(); acc[3] += dato[~t].sum()
LIFT = pd.DataFrame(lift)
pool = pd.DataFrame([dict(variable=v, pct_top=100 * a / max(1, b), pct_resto=100 * c / max(1, d))
                     for v, (a, b, c, d) in filas_pool.items()])
pool["razon"] = pool["pct_top"] / pool["pct_resto"].replace(0, np.nan)
say("\n(6b) Señal activa en el 1 % superior frente al resto, todos los grupos juntos")
say("     (activa = el fenómeno ocurre, la bandera vale 1, o el percentil está en el 10 % más alto)")
say(pool.sort_values("razon", ascending=False).round(2).to_string(index=False))
say("\nRazón (top / resto) por grupo (vacío = la variable no se usa en ese grupo):")
say(LIFT.pivot_table(index="grupo", columns="variable", values="razon").rename(columns=CORTO)
      .round(1).to_string())
for _, f in LIFT[(LIFT["pct_top"] >= 80) & (LIFT["pct_resto"] <= 20)].iterrows():
    ALERTAS.append(f"{f.grupo}: el {f.pct_top:.0f} % del 1 % superior tiene '{f.variable}' activa "
                   f"(resto {f.pct_resto:.1f} %): la lista puede estar dominada por esa señal.")

# Frecuencia de cada variable entre las tres que explican cada proceso priorizado
ex = RES.loc[RES["top_ensamble"], "explica"].str.split(" | ", regex=False).explode().str.rsplit(" ", n=1).str[0]
say("\nVariables que más aparecen entre las tres que explican cada proceso priorizado (% de procesos):")
say((100 * ex.value_counts() / max(1, int(RES["top_ensamble"].sum()))).round(1).to_string())

# (6c) Monto
say("\n(6c) Monto: mediana del 1 % superior frente al resto y parte del valor del grupo")
tp, rs_ = RES[RES["top_ensamble"]], RES[~RES["top_ensamble"]]
mon = pd.DataFrame({
    "mediana_top": tp.groupby("grupo")["monto_final"].median(),
    "mediana_resto": rs_.groupby("grupo")["monto_final"].median(),
    "pct_valor_en_top": 100 * tp.groupby("grupo")["monto_final"].sum() / RES.groupby("grupo")["monto_final"].sum()})
say(mon.round(1).to_string())

# (6d) Años
say("\n(6d) Años (período anterior a la reforma): participación en el 1 % superior / participación en el grupo")
pre = RES[RES["periodo_calc"] == "pre"]
anios = pre.dropna(subset=["anio_proc"]).groupby("anio_proc").agg(procesos=("ocid", "size"), top=("top_ensamble", "sum"))
anios.index = anios.index.astype(int)
anios["razon"] = (anios["top"] / anios["top"].sum()) / (anios["procesos"] / anios["procesos"].sum())
say(anios.round(2).T.to_string())
fuera_rango = anios[(anios["procesos"] >= 1000) & ((anios["razon"] > 1.5) | (anios["razon"] < 0.67))]
if len(fuera_rango):
    ALERTAS.append("El 1 % superior se concentra o escasea en ciertos años: "
                   + ", ".join(f"{int(a)} ({x:.2f})" for a, x in fuera_rango["razon"].items()))

# (6e) Concentración (sin identidades)
say("\n(6e) Concentración del 1 % superior en entidades y proveedores (solo conteos, sin RUC)")
conc2 = []
for g, r in RES.groupby("grupo"):
    t = r[r.top_ensamble]
    fila = dict(grupo=g, top=len(t))
    for col, nom in [("entidad_ruc", "entidades"), ("proveedor_ruc_final", "proveedores")]:
        if col in r.columns and len(t):
            vc = t[col].value_counts()
            fila[f"{nom}_distintas"] = len(vc)
            fila[f"{nom}_max_procesos"] = int(vc.iloc[0])
            fila[f"{nom}_pct_10_mayores"] = round(100 * vc.head(10).sum() / len(t), 1)
    conc2.append(fila)
say(pd.DataFrame(conc2).to_string(index=False))

# (6f) Modelos grandes y pequeños
say("\n(6f) Procesos puntuados con modelos de patrones pequeños: peso en el grupo y en el 1 % superior")
f6 = []
for g, r in RES.groupby("grupo"):
    peq = ~r["modelo"].isin(PLAN[g]["grandes"]) & (r["modelo"] != "")
    sin = r["modelo"] == ""
    pt = 100 * peq[r.top_ensamble].mean() if r.top_ensamble.any() else np.nan
    f6.append(dict(grupo=g, modelos=len(PLAN[g]["vars_modelo"]), pct_procesos_pequenos=100 * peq.mean(),
                   pct_top_pequenos=pt, procesos_sin_puntaje=int(sin.sum())))
F6 = pd.DataFrame(f6)
say(F6.round(2).to_string(index=False))
for _, f in F6.iterrows():
    top_g = int(RES.loc[RES["grupo"] == f.grupo, "top_ensamble"].sum())
    if top_g >= 20 and f.pct_procesos_pequenos > 0 and f.pct_top_pequenos > 3 * max(f.pct_procesos_pequenos, 0.33):
        ALERTAS.append(f"{f.grupo}: los modelos de patrones pequeños están sobrerrepresentados en el 1 % superior.")

CONC.to_csv(os.path.join(DIR_RUN, "6a_concordancia.csv"), index=False)
LIFT.to_csv(os.path.join(DIR_RUN, "6b_senales_top.csv"), index=False)
say(f"[celda 6: {time.time() - t_celda:.0f} s]")

In [ ]:
# =====================================================================================
# CELDA 7. Estabilidad: se reajusta con otras semillas y se compara la lista del 1 % superior
# =====================================================================================
empezar("celda 7")
t_celda = time.time()
SEMILLAS_ESTAB = [11, 22]           # en la corrida final del anteproyecto serán 20
GRUPOS_ESTAB = [g for g, p in PLAN.items() if len(p["idx"]) >= 5000]
filas = []
for g in GRUPOS_ESTAB:
    base_r = RES.loc[PLAN[g]["idx"]]
    for s in SEMILLAS_ESTAB:
        t = time.time()
        r2, _, _ = puntuar_grupo(PLAN[g], X.loc[PLAN[g]["idx"]], s)
        r2 = marcar_tops(r2, s)
        fila = dict(grupo=g, semilla=s)
        for m in ["ensamble", "if", "lof", "ae"]:
            fila[f"jac_{m}"] = round(jaccard(base_r[f"top_{m}"], r2[f"top_{m}"]), 3)
        fila["jac_consenso"] = round(jaccard(base_r["top_ensamble"] & base_r["consenso"],
                                             r2["top_ensamble"] & r2["consenso"]), 3)
        fila["seg"] = round(time.time() - t, 1)
        filas.append(fila)
        say(f"  {g:<42} semilla {s:<3} jaccard ensamble {fila['jac_ensamble']:.3f}  ({fila['seg']} s)")
ESTAB = pd.DataFrame(filas)
if len(ESTAB):
    say("\nJaccard del 1 % superior frente a la corrida principal (1 = misma lista):")
    say(ESTAB.groupby("grupo")[["jac_ensamble", "jac_consenso", "jac_if", "jac_lof", "jac_ae"]]
          .mean().round(3).to_string())
    for g, j in ESTAB.groupby("grupo")["jac_ensamble"].mean().items():
        if j < 0.6:
            ALERTAS.append(f"{g}: la lista cambia mucho con la semilla (Jaccard medio {j:.2f}).")
    ESTAB.to_csv(os.path.join(DIR_RUN, "7_estabilidad.csv"), index=False)
say(f"[celda 7: {time.time() - t_celda:.0f} s]")

In [ ]:
# =====================================================================================
# CELDA 8. Piloto de H1: anomalías inyectadas en procesos de 2023
#   Comprueba que la maquinaria de H1 funciona y da órdenes de magnitud. Usa 2023 a propósito:
#   2024 queda intacto para la evaluación final de H1, que no debe usarse para tomar decisiones.
#   Aquí se inyecta directamente en la escala de percentil; la evaluación final inyecta en las
#   variables de origen, recalcula los indicadores y usa 20 semillas.
#   En cada grupo se altera el 1 % de los procesos del año, repartido entre los tipos, y se mide
#   cuántos quedan dentro del 1 % superior de los procesos de ese año (al azar: un 1 %).
# =====================================================================================
empezar("celda 8")
t_celda = time.time()
H1_ANIO = 2023
H1_SEMILLAS = [1, 2, 3]
H1_MIN_ANIO = 300 if MODO == "RAPIDO" else 1000   # procesos del año que debe tener el grupo
H1_MAX_ENTRENO = 30_000                           # el año completo + muestra de otros años
COMPETITIVAS = ("subasta", "menor cuantia", "cotizacion", "licitacion", "lista corta", "concurso")

def tipos_h1(g, cols):
    """Tipos de anomalía: cambios en la escala de percentil (1 = extremo) que se aplican."""
    t = {"precio_p95": {"precio_alto": 0.95},
         "precio_p99": {"precio_alto": 0.99},
         "precio_p999": {"precio_alto": 0.999},
         "nuevo_y_capital": {"proveedor_nuevo": 0.99, "monto_capital": 0.99, "debutante": 1.0},
         "combinado_moderado": {"precio_alto": 0.90, "proveedor_nuevo": 0.90, "debutante": 1.0}}
    if sin_tildes(g).startswith(COMPETITIVAS) and "oferente_unico" in cols:
        t["oferente_unico"] = {"oferente_unico": 1.0, "poca_concurrencia": 1.0}
        t["combinado_moderado"].update({"oferente_unico": 1.0, "poca_concurrencia": 1.0})
    if "plazo_corto" in cols:
        t["plazo_p95"] = {"plazo_corto": 0.95}
        t["plazo_p99"] = {"plazo_corto": 0.99}
    t = {k: {v: x for v, x in c.items() if v in cols} for k, c in t.items()}
    return {k: c for k, c in t.items() if any(TIPO[v] == "rango" or v == "oferente_unico" for v in c)}

def inyectar(Xg, filas_ok, cambios):
    """Aplica los cambios y mantiene la coherencia: un debutante no tiene historial previo."""
    for v, val in cambios.items():
        Xg.loc[filas_ok, v] = val
    if cambios.get("debutante") == 1.0:
        for v, val in [("primera_vez_ent", 1.0), ("cuota_entidad", 0.0), ("recurrencia_par", 0.0)]:
            if v in Xg.columns:
                Xg.loc[filas_ok, v] = val
    if cambios.get("oferente_unico") == 1.0 and "poca_concurrencia" in Xg.columns:
        Xg.loc[filas_ok, "poca_concurrencia"] = 1.0

filas, posiciones = [], []
for g, plan in PLAN.items():
    if not g.endswith("| pre"):
        continue
    idx = plan["idx"]
    es_anio = (df.loc[idx, "anio_proc"] == H1_ANIO).to_numpy()
    if es_anio.sum() < H1_MIN_ANIO:
        continue
    for s in H1_SEMILLAS:
        rng = np.random.default_rng(1000 + s)
        i_anio, otros = idx[es_anio], idx[~es_anio]
        n_otros = max(0, min(len(otros), H1_MAX_ENTRENO - len(i_anio)))
        tr_idx = i_anio.append(pd.Index(rng.choice(otros, size=n_otros, replace=False)))
        Xg = X.loc[tr_idx, plan["usar"]].copy()
        tipos = tipos_h1(g, plan["usar"])
        n_iny = max(1, int(round(CORTE_TOP * len(i_anio))))
        orden = list(rng.permutation(list(tipos)))
        cuantos = pd.Series([orden[i % len(orden)] for i in range(n_iny)]).value_counts()
        libres = pd.Index(rng.permutation(i_anio))
        marca_tipo = pd.Series("", index=tr_idx)
        for tipo, cambios in tipos.items():
            if tipo not in cuantos.index:
                continue
            X24 = Xg.loc[libres]
            ok = pd.Series(True, index=libres)
            for v, val in cambios.items():
                if TIPO[v] == "rango":             # que tenga el dato y que la inyección lo suba
                    ok &= X24[v].notna() & (X24[v] < val)
            if tipo == "oferente_unico":
                ok &= X24["oferente_unico"] == 0
            elegidos = libres[ok.to_numpy()][:int(cuantos[tipo])]
            inyectar(Xg, elegidos, cambios)
            marca_tipo.loc[elegidos] = tipo
            libres = libres.difference(elegidos, sort=False)
        plan_h1 = dict(plan)
        plan_h1["clave"] = plan["clave"].loc[tr_idx]
        r, _, _ = puntuar_grupo(plan_h1, Xg, SEMILLA + s)
        r = r.loc[i_anio]
        mt = marca_tipo.loc[i_anio]
        iny = mt != ""
        for metodo in ["ensamble", "if", "lof", "ae", "base"]:
            pctl = r[metodo].rank(pct=True)                 # posición dentro del año (1 = más anómalo)
            for frac in [0.01, 0.02, 0.05]:
                top = pd.Series(top_k(r[metodo], frac, s), index=r.index)
                for tipo in tipos:
                    sel = mt == tipo
                    if sel.sum():
                        filas.append(dict(grupo=g, semilla=s, metodo=metodo, corte=frac, tipo=tipo,
                                          inyectados=int(sel.sum()), recuperados=int(top[sel].sum())))
            posiciones.append(pd.DataFrame(dict(grupo=g, semilla=s, metodo=metodo, tipo=mt[iny].values,
                                                ocid_idx=mt[iny].index, percentil=pctl[iny].values,
                                                en_top1=top_k(r[metodo], 0.01, s)[iny.to_numpy()])))
        say(f"  {g:<42} semilla {s}: {int(iny.sum())} inyectados en {len(i_anio):,} procesos de {H1_ANIO}")

H1 = pd.DataFrame(filas)
if len(H1):
    POS = pd.concat(posiciones, ignore_index=True)
    agg = H1.groupby(["corte", "tipo", "metodo"])[["inyectados", "recuperados"]].sum().reset_index()
    agg["recuperacion"] = agg["recuperados"] / agg["inyectados"]
    t1 = agg[agg["corte"] == 0.01].pivot_table(index="tipo", columns="metodo", values="recuperacion")
    t1.insert(0, "inyectados", agg[(agg["corte"] == 0.01) & (agg["metodo"] == "ensamble")].set_index("tipo")["inyectados"])
    say("\n(8a) Recuperación en el 1 % superior por tipo de anomalía (al azar: 0,01)")
    say(t1[["inyectados", "ensamble", "base", "if", "lof", "ae"]].round(3).to_string())
    t2 = agg.groupby(["corte", "metodo"])[["inyectados", "recuperados"]].sum()
    t2["recuperacion"] = (t2["recuperados"] / t2["inyectados"]).round(3)
    say("\n(8b) Recuperación global según el corte (al azar: igual al corte)")
    say(t2["recuperacion"].unstack("metodo").to_string())
    t3 = H1[H1["corte"] == 0.01].groupby(["grupo", "metodo"])[["inyectados", "recuperados"]].sum()
    t3["recuperacion"] = (t3["recuperados"] / t3["inyectados"]).round(3)
    say("\n(8c) Recuperación en el 1 % superior por grupo")
    say(t3["recuperacion"].unstack("metodo").to_string())
    say(f"\n(8d) Percentil mediano de los procesos inyectados dentro de {H1_ANIO} (1 = el más anómalo)")
    say(POS.pivot_table(index="tipo", columns="metodo", values="percentil", aggfunc="median")
          [["ensamble", "base", "if", "lof", "ae"]].round(3).to_string())
    # diferencia pareada ensamble - índice simple, con remuestreo de los procesos inyectados
    par = POS.pivot_table(index=["grupo", "semilla", "ocid_idx"], columns="metodo", values="en_top1")
    dif = (par["ensamble"].astype(float) - par["base"].astype(float)).to_numpy()
    bs = np.random.default_rng(SEMILLA).choice(dif, size=(2000, len(dif)), replace=True).mean(axis=1)
    lo_, hi_ = np.percentile(bs, [2.5, 97.5])
    say(f"\n(8e) Ensamble menos índice simple, recuperación en el 1 % superior: {dif.mean():+.3f} "
          f"(IC 95 % por remuestreo: {lo_:+.3f} a {hi_:+.3f}; {len(dif)} inyecciones)")
    H1_RES = dict(rec_ensamble=float(par["ensamble"].mean()), rec_base=float(par["base"].mean()),
                  dif=float(dif.mean()), ic=[float(lo_), float(hi_)], n=int(len(dif)))
    if dif.mean() <= 0:
        ALERTAS.append(f"Piloto H1 ({H1_ANIO}): el ensamble no supera al índice simple ({dif.mean():+.3f}).")
    H1.to_csv(os.path.join(DIR_RUN, "8_piloto_h1.csv"), index=False)
    POS.to_csv(os.path.join(DIR_RUN, "8_piloto_h1_posiciones.csv"), index=False)
else:
    H1_RES = {}
    say(f"Ningún grupo tiene suficientes procesos de {H1_ANIO} en esta corrida.")
    ALERTAS.append(f"El piloto de H1 no corrió (pocos procesos de {H1_ANIO}).")
say(f"[celda 8: {time.time() - t_celda:.0f} s]")

In [ ]:
# =====================================================================================
# CELDA 9. Tablero final, alertas y registro de la corrida (esto es lo más importante para pegar)
# =====================================================================================
empezar("celda 9")
ALERTAS[:] = list(dict.fromkeys(ALERTAS))          # sin repetidas si alguna celda se corrió dos veces
tablero = CONC.set_index("grupo")[["procesos", "top", "W_kendall", "jac_if_lof", "jac_if_ae", "jac_lof_ae",
                                   "consenso_pct", "rho_ens_base", "rho_ens_nvars", "rho_ens_monto"]].copy()
tablero.insert(1, "modelos", pd.Series({g: len(p["vars_modelo"]) for g, p in PLAN.items()}))
if len(ESTAB):
    tablero["estab_jac"] = ESTAB.groupby("grupo")["jac_ensamble"].mean()
if len(H1):
    rec = H1[H1["corte"] == 0.01].groupby(["grupo", "metodo"])[["inyectados", "recuperados"]].sum()
    rec = (rec["recuperados"] / rec["inyectados"]).unstack("metodo")
    tablero["h1_ens"], tablero["h1_base"] = rec["ensamble"], rec["base"]
tablero.to_csv(os.path.join(DIR_RUN, "9_tablero.csv"))

w = CONC["procesos"] / CONC["procesos"].sum()
pct_top = pool.set_index("variable")["pct_top"]
FICHA = dict(representacion=REPRESENTACION,
             piloto_ensamble=H1_RES.get("rec_ensamble"), piloto_indice=H1_RES.get("rec_base"),
             lista_con_recompra_pct=float(pct_top.get("recompra_30d", np.nan)),
             lista_con_precio_alto_pct=float(pct_top.get("precio_alto", np.nan)),
             consenso_pct=float(np.nansum(CONC["consenso_pct"] * w)),
             estabilidad_jaccard=float(ESTAB["jac_ensamble"].mean()) if len(ESTAB) else None)

resumen = {
    "corrida": RUN_ID, "modo": MODO, "ficha": FICHA, "procesos": int(len(RES)), "grupos_modelados": len(PLAN),
    "grupos_no_modelados": NO_MODELADOS,
    "huella_indicadores": HUELLA_IX, "huella_procesos": HUELLA_PM,
    "parametros": dict(IF_ARBOLES=IF_ARBOLES, IF_SUBMUESTRA=IF_SUBMUESTRA, LOF_VECINOS=LOF_VECINOS,
                       CORTE_TOP=CORTE_TOP, MIN_PATRON=MIN_PATRON, MIN_GRUPO=MIN_GRUPO, REF_MAX=REF_MAX,
                       AE_MAX_ENTRENO=AE_MAX_ENTRENO, HUERFANO_MAX=HUERFANO_MAX, SEMILLA=SEMILLA),
    "versiones": dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,
                      sklearn=sklearn.__version__, duckdb=duckdb.__version__),
    "variables": list(X.columns),
    "lista_priorizada": int(RES["priorizado"].sum()),
    "nucleo_consenso": int((RES.top_ensamble & RES.consenso).sum()),
    "representacion": REPRESENTACION, "anio_inicio": ANIO_INICIO,
    "definiciones": {"debutante": "sin procesos del proveedor en los 365 días anteriores",
                     "primera_vez_ent": "fuera de los modelos (igual a recurrencia = 0)"},
    "piloto_h1_anio": H1_ANIO, "piloto_h1": H1_RES,
    "minutos": round((time.time() - T_INICIO) / 60, 1),
    "alertas": ALERTAS,
}
with open(os.path.join(DIR_RUN, "registro_corrida.json"), "w", encoding="utf-8") as f:
    json.dump(resumen, f, ensure_ascii=False, indent=1)

say("=" * 110)
say(f"TABLERO DE LA CORRIDA {RUN_ID}  ({resumen['minutos']} min)")
say("=" * 110)
say(tablero.round(3).to_string())
say("\nPromedios ponderados por procesos: " + " | ".join(
    f"{c} {np.nansum(CONC[c] * w):.3f}" for c in ["W_kendall", "jac_if_lof", "jac_if_ae", "jac_lof_ae",
                                                    "consenso_pct", "rho_ens_base", "rho_ens_nvars"]))
if H1_RES:
    say(f"Piloto H1 en {H1_ANIO} (1 % superior): ensamble {H1_RES['rec_ensamble']:.3f} | índice simple "
          f"{H1_RES['rec_base']:.3f} | diferencia {H1_RES['dif']:+.3f} "
          f"(IC 95 % {H1_RES['ic'][0]:+.3f} a {H1_RES['ic'][1]:+.3f})")
def _f(x, fmt):
    return "s/d" if x is None or (isinstance(x, float) and not np.isfinite(x)) else format(x, fmt)
say(f"\nFICHA ({REPRESENTACION}): piloto H1 ensamble {_f(FICHA['piloto_ensamble'], '.3f')} vs índice "
    f"{_f(FICHA['piloto_indice'], '.3f')} | lista con recompra {_f(FICHA['lista_con_recompra_pct'], '.0f')} %"
    f" | con precio alto {_f(FICHA['lista_con_precio_alto_pct'], '.0f')} % | consenso "
    f"{_f(FICHA['consenso_pct'], '.0f')} % | estabilidad {_f(FICHA['estabilidad_jaccard'], '.2f')}")
say(f"\nALERTAS ({len(ALERTAS)}):" if ALERTAS else "\nALERTAS: ninguna")
for a in ALERTAS:
    say("  -", a)
say("\nArchivos guardados en", DIR_RUN)
for f in sorted(os.listdir(DIR_RUN)):
    say("  ", f, f"({os.path.getsize(os.path.join(DIR_RUN, f)) / 1e6:.1f} MB)")
guardar_txt()
say("\nAdjunta en el chat el archivo resumen_para_chat.txt de esa carpeta.")
guardar_txt()

In [ ]:
# Enmienda B (16-oct-2026): reglas de H2 y H3 + congelamiento de la lista priorizada.
# Se corre UNA vez. Solo escribe un archivo de constancia; no lee ninguna marca del SRI.
import os, json, hashlib
from datetime import datetime

ENMIENDAS = r"C:\SERCOP_FINAL\enmiendas"
RUN = r"C:\SERCOP_FINAL\modelos\20261016_1017_completo_normal_sup"   # la corrida COMPLETO

def huella(path, n=16):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()[:n]

previa = os.path.join(ENMIENDAS, "enmienda_2026-10-16.json")
congelados = {f: huella(os.path.join(RUN, f)) for f in ["puntajes.parquet", "registro_corrida.json"]}

enm = {
    "fecha": datetime.now().isoformat(timespec="seconds"),
    "id": "enmienda_2026-10-16_b",
    "sigue_a": {"archivo": "enmienda_2026-10-16.json", "huella": huella(previa)},
    "motivo": ("La corrida COMPLETO (normal_sup, desde 2016) sobrerrepresenta 2016-2017 en la lista "
               "(razón 1,42 y 1,43) y la subrepresenta desde 2020 (0,69 a 0,87), por la caída del historial "
               "proveedor-entidad y de la recompra en el tiempo; el puntaje correlaciona con el monto "
               "(0,14 a 0,30); y la lista se concentra en pocos proveedores (un proveedor con 59 de 570 "
               "priorizados en régimen especial). Se decide sin haber consultado ninguna marca del SRI."),
    "decisiones": {
        "H2_principal": ("Percentil del puntaje del ensamble calculado dentro de grupo x año (año de "
                         "fecha_inicio_tender); estadístico: percentil mediano > 0,5; permutación por "
                         "proveedor con 9.999 réplicas; prueba unilateral, Holm junto a H3."),
        "H2_sensibilidad": "Mismo cálculo dentro de grupo x año x decil de monto (decil dentro de grupo x año).",
        "H2_poblacion": ("Solo procesos puntuados (desde 2016). Los procesos de 2015 de los contribuyentes "
                         "del SRI quedan fuera; se declara sin haber contado cuántos son."),
        "H3_muestra": ("60 priorizados (columna 'priorizado') y 60 controles del mismo grupo, año y decil de "
                       "monto, con un máximo de 2 procesos por proveedor y 2 por entidad en cada lado; "
                       "sorteo con semilla 20261001."),
        "H3_nota": ("Con el tope la muestra no reproduce exactamente la composición de la lista; a cambio, "
                    "mide casos distintos. Se reporta así en la tesis."),
    },
    "lista_congelada": {"corrida": os.path.basename(RUN.rstrip("\\/")), "huellas": congelados},
}
arch = os.path.join(ENMIENDAS, "enmienda_2026-10-16_b.json")
if os.path.exists(arch):
    raise SystemExit(f"{arch} ya existe: no se sobrescribe una enmienda. Avísame si hay que corregirla.")
with open(arch, "w", encoding="utf-8") as f:
    json.dump(enm, f, ensure_ascii=False, indent=1)
print("Enmienda B guardada en", arch, "| huella", huella(arch))
print("Lista congelada:", congelados)

In [ ]:
# =====================================================================================
# CELDA R. ¿Se pueden recalcular los indicadores desde sus columnas de origen?
#   La evaluación final de H1 altera las variables de origen (precio, oferentes, plazo, antigüedad,
#   capital) y después recalcula los indicadores con la MISMA regla con que se construyó la base.
#   Esta celda descubre y verifica esa regla. No usa puntajes de los modelos ni marcas del SRI.
#   Se corre una vez; guarda reproduccion_indicadores.json, que leerá el cuaderno 17.
# =====================================================================================
import os, json, hashlib, unicodedata, itertools, time
from datetime import datetime
import numpy as np
import pandas as pd
import duckdb

BASE = r"C:\SERCOP_FINAL\plan_A"
PREREG = r"C:\SERCOP_FINAL\preregistro"
os.makedirs(PREREG, exist_ok=True)
t0 = time.time()

def ruta(nombre):
    return os.path.join(BASE, nombre).replace("\\", "/")

def huella(path, n=16):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()[:n]

def num(s):
    """Cualquier columna a float (booleanos y nulos incluidos)."""
    try:
        return pd.Series(pd.array(s, dtype="Float64").to_numpy(dtype="float64", na_value=np.nan), index=s.index)
    except Exception:
        return pd.to_numeric(s.astype("object"), errors="coerce").astype("float64")

def sin_tildes(x):
    t = unicodedata.normalize("NFKD", str(x))
    return "".join(c for c in t if not unicodedata.combining(c)).lower()

IX = ruta("planA_indicadores.parquet")
con = duckdb.connect()
cols = [r[0] for r in con.execute(f"DESCRIBE SELECT * FROM read_parquet('{IX}')").fetchall()]
quiero = ["ocid", "familia", "periodo", "anio", "fecha_proceso", "cpc5", "unidad", "precio_unit",
          "ind_precio_nivel", "ind_precio_n_grupo", "ind_desv_precio_unit_z", "ind_monto_sobre_capital",
          "ind_antiguedad_dias", "ind_plazo_ofertas_dias", "ind_plazo_inicio_adj_dias",
          "ind_plazo_adj_firma_dias", "ind_fraccionamiento_monto_30d", "ind_fraccionamiento_n_30d",
          "ind_fraccionamiento_marca", "ind_cuota_prov_en_entidad_12m", "ind_recurrencia_par_12m",
          "ind_n_oferentes", "ind_oferente_unico", "ref_concurrencia_grupo", "dif_concurrencia_grupo",
          "pct_precio_alto", "pct_monto_capital", "pct_proveedor_nuevo", "pct_plazo_corto",
          "pct_firma_rapida", "pct_fraccionamiento_pos", "pct_cuota_pos", "pct_recurrencia_pos"]
faltan = [c for c in quiero if c not in cols]
print("Columnas que no están (se omiten sus pruebas):", faltan if faltan else "ninguna")
d = con.execute(f"SELECT {', '.join(c for c in quiero if c in cols)} FROM read_parquet('{IX}')").df()
fecha = pd.to_datetime(d["fecha_proceso"], errors="coerce")
d["anio_fecha"] = fecha.dt.year
d["grupo"] = d["familia"].astype(str) + " | " + d["periodo"].astype(str)
print(f"Procesos: {len(d):,} | grupos familia x periodo: {d['grupo'].nunique()}")

# -------------------------------------------------------------------------------------
# (A) Percentiles: ¿qué columna, qué orientación, qué agrupación y qué fórmula de empates?
# -------------------------------------------------------------------------------------
CANDIDATOS = {
    "pct_precio_alto":         [("ind_desv_precio_unit_z", +1, "todos")],
    "pct_monto_capital":       [("ind_monto_sobre_capital", +1, "todos")],
    "pct_proveedor_nuevo":     [("ind_antiguedad_dias", -1, "todos")],
    "pct_plazo_corto":         [("ind_plazo_ofertas_dias", -1, "todos"), ("ind_plazo_inicio_adj_dias", -1, "todos")],
    "pct_firma_rapida":        [("ind_plazo_adj_firma_dias", -1, "todos")],
    "pct_fraccionamiento_pos": [("ind_fraccionamiento_monto_30d", +1, "positivos"),
                                ("ind_fraccionamiento_n_30d", +1, "positivos")],
    "pct_cuota_pos":           [("ind_cuota_prov_en_entidad_12m", +1, "positivos")],
    "pct_recurrencia_pos":     [("ind_recurrencia_par_12m", +1, "positivos")],
}
AGRUPACIONES = {"familia|periodo": d["grupo"], "familia": d["familia"].astype(str)}

def formulas(x, g):
    """Cuatro formas habituales de convertir x en percentil dentro de g (vacíos quedan vacíos)."""
    gb = x.groupby(g)
    r_avg, r_min, r_max = gb.rank(method="average"), gb.rank(method="min"), gb.rank(method="max")
    n = x.notna().groupby(g).transform("sum")
    return {"mitad_de_salto": (r_avg - 0.5) / n, "rango_medio/n": r_avg / n,
            "acumulada (max/n)": r_max / n, "percent_rank": (r_min - 1) / (n - 1)}

REGLAS, filas = {}, []
for pct, cands in CANDIDATOS.items():
    if pct not in d.columns:
        continue
    real = num(d[pct])
    mejor = None
    for (fuente, signo, sub) in cands:
        if fuente not in d.columns:
            continue
        x = signo * num(d[fuente])
        if sub == "positivos":
            x = x.where(num(d[fuente]) > 0)
        for nom_g, g in AGRUPACIONES.items():
            for nom_f, pred in formulas(x, g).items():
                ok = real.notna() & pred.notna()
                igual = float(((real[ok] - pred[ok]).abs() < 1e-6).mean()) if ok.any() else 0.0
                cob = int((real.notna() != pred.notna()).sum())
                filas.append(dict(indicador=pct, fuente=fuente, signo=signo, subconjunto=sub, agrupacion=nom_g,
                                  formula=nom_f, coincide_pct=round(100 * igual, 3), cobertura_distinta=cob))
                cand = (igual, -cob)
                if mejor is None or cand > mejor[0]:
                    mejor = (cand, dict(fuente=fuente, signo=signo, subconjunto=sub, agrupacion=nom_g,
                                        formula=nom_f, coincide_pct=round(100 * igual, 3), cobertura_distinta=cob))
    if mejor:
        REGLAS[pct] = mejor[1]
TAB_A = pd.DataFrame(filas)
print("\n(A) Mejor regla encontrada para cada percentil:")
print(pd.DataFrame(REGLAS).T[["fuente", "signo", "subconjunto", "agrupacion", "formula", "coincide_pct",
                              "cobertura_distinta"]].to_string())

# -------------------------------------------------------------------------------------
# (B) Oferente único y concurrencia
# -------------------------------------------------------------------------------------
print("\n(B) Concurrencia:")
CONC = {}
if {"ind_oferente_unico", "ind_n_oferentes"} <= set(d.columns):
    n_of, ou = num(d["ind_n_oferentes"]), num(d["ind_oferente_unico"])
    ok = n_of.notna() & ou.notna()
    CONC["oferente_unico_es_n_igual_1_pct"] = round(100 * float((ou[ok] == (n_of[ok] == 1)).mean()), 3)
    print(f"  oferente único = (número de oferentes == 1): {CONC['oferente_unico_es_n_igual_1_pct']} %")
if {"dif_concurrencia_grupo", "ref_concurrencia_grupo", "ind_n_oferentes"} <= set(d.columns):
    dif, ref = num(d["dif_concurrencia_grupo"]), num(d["ref_concurrencia_grupo"])
    ok = dif.notna() & ref.notna()
    CONC["dif_es_ref_menos_n_pct"] = round(100 * float(((ref[ok] - n_of[ok]) - dif[ok]).abs().lt(1e-6).mean()), 3)
    CONC["dif_vacia_si_un_oferente_pct"] = round(100 * float(dif[n_of == 1].isna().mean()), 3) if (n_of == 1).any() else None
    print(f"  dif_concurrencia = referencia - oferentes: {CONC['dif_es_ref_menos_n_pct']} % | "
          f"vacía cuando hay un solo oferente: {CONC['dif_vacia_si_un_oferente_pct']} %")

# -------------------------------------------------------------------------------------
# (C) Precio: grupo de referencia de cada proceso, centro y escala del z robusto
# -------------------------------------------------------------------------------------
print("\n(C) Precio unitario:")
PRECIO = {"ok": False}
if {"precio_unit", "ind_precio_nivel", "ind_desv_precio_unit_z", "cpc5"} <= set(d.columns):
    pu = num(d["precio_unit"])
    d["lp"] = np.log(pu.where(pu > 0))
    d["z"] = num(d["ind_desv_precio_unit_z"])
    c5 = d["cpc5"].astype(str).str.replace(r"\.0$", "", regex=True).str.replace(r"\D", "", regex=True)
    d["cpc5_s"] = c5.where(c5 != "", np.nan).str.zfill(5)
    d["cpc3_s"] = d["cpc5_s"].str[:3]
    d["unidad_s"] = d["unidad"].astype(str).str.strip().str.upper() if "unidad" in d.columns else "-"
    d["unidad_raw"] = d["unidad"] if "unidad" in d.columns else "-"
    d["familia_s"] = d["familia"].astype(str)
    niv = d["ind_precio_nivel"].astype("object")
    print("  Niveles de referencia en la base:", niv.value_counts(dropna=False).to_dict())

    def claves_base(etiqueta):
        """Claves posibles del grupo de referencia según la etiqueta del nivel."""
        e = sin_tildes(etiqueta)
        if "unid" in e:
            return [["cpc5_s", "unidad_s"], ["cpc5_s", "unidad_raw"]]
        if "3" in e:
            return [["cpc3_s"]]
        if "5" in e or "cpc" in e:
            return [["cpc5_s"]]
        if "fam" in e or "glob" in e:
            return [[]]
        return None

    ng = num(d["ind_precio_n_grupo"]) if "ind_precio_n_grupo" in d.columns else None
    tiene_lp = d["lp"].notna()
    elegidas, filas_c = {}, []
    for etiqueta in niv.dropna().unique():
        bases = claves_base(etiqueta)
        if bases is None:
            continue
        es = (niv == etiqueta)
        mejor = None
        for base_k, extra in itertools.product(bases, [[], ["familia_s"], ["anio"], ["familia_s", "anio"],
                                                       ["anio_fecha"], ["familia_s", "anio_fecha"]]):
            keys = base_k + extra
            if not keys or any(k not in d.columns for k in keys):
                continue
            for alcance in ["todas", "mismo_nivel"]:
                m = tiene_lp & (es if alcance == "mismo_nivel" else True)
                tam = d.loc[m].groupby(keys, dropna=False)["lp"].transform("size").reindex(d.index)
                coinc = float((tam[es & tiene_lp] == ng[es & tiene_lp]).mean()) if ng is not None else 0.0
                filas_c.append(dict(nivel=etiqueta, claves="+".join(keys), alcance=alcance,
                                    tamano_coincide_pct=round(100 * coinc, 2)))
                # desempate: más coincidencia, luego más claves (más específico), luego "todas"
                orden = (round(coinc, 6), len(keys), alcance == "todas")
                if mejor is None or orden > mejor[3]:
                    mejor = (coinc, keys, alcance, orden)
        elegidas[etiqueta] = mejor
    print(pd.DataFrame(filas_c).sort_values(["nivel", "tamano_coincide_pct"], ascending=[True, False])
          .groupby("nivel").head(3).to_string(index=False))

    # centro y escala: z = (lp - centro) / escala; se prueba centro = mediana y escala = c * MAD
    d["m_ref"], d["mad_ref"] = np.nan, np.nan
    for etiqueta, (coinc, keys, alcance, _) in elegidas.items():
        es = (niv == etiqueta)
        m = tiene_lp & (es if alcance == "mismo_nivel" else True)
        sub = d.loc[m, keys + ["lp"]].copy()
        gb = sub.groupby(keys, dropna=False)["lp"]
        med = gb.transform("median")
        mad = (sub["lp"] - med).abs().groupby([sub[k] for k in keys], dropna=False).transform("median")
        med, mad = med.reindex(d.index), mad.reindex(d.index)
        d.loc[es, "m_ref"], d.loc[es, "mad_ref"] = med[es], mad[es]
    ok = d["z"].notna() & d["lp"].notna() & d["mad_ref"].gt(0) & d["z"].abs().gt(0.05)
    c_i = ((d.loc[ok, "lp"] - d.loc[ok, "m_ref"]) / d.loc[ok, "z"]) / d.loc[ok, "mad_ref"]
    c_est = float(np.nanmedian(c_i)) if ok.any() else np.nan
    z_hat = (d["lp"] - d["m_ref"]) / (c_est * d["mad_ref"])
    okz = d["z"].notna() & z_hat.notna()
    rep = (z_hat[okz] - d.loc[okz, "z"]).abs() < 1e-6 * np.maximum(1, d.loc[okz, "z"].abs())
    pct_rep = 100 * float(rep.mean()) if okz.any() else 0.0
    d["precio_reproducido"] = False
    d.loc[rep.index[rep], "precio_reproducido"] = True
    print(f"  Escala estimada: z = (ln precio - mediana del grupo) / ({c_est:.4f} x MAD)")
    print(f"  z reproducido en el {pct_rep:.2f} % de {int(okz.sum()):,} procesos con z")
    por_fam = d.loc[d["z"].notna()].groupby("familia_s")["precio_reproducido"].mean().mul(100).round(2)
    print("  Por familia (% reproducido):", por_fam.to_dict())
    PRECIO = dict(ok=pct_rep >= 99, pct_reproducido=round(pct_rep, 3), c_mad=c_est,
                  niveles={str(k): dict(claves=v[1], alcance=v[2], tamano_coincide_pct=round(100 * v[0], 2))
                           for k, v in elegidas.items()})

# -------------------------------------------------------------------------------------
# (D) Magnitudes de H1 (solo distribución de variables, procesos de 2024 anteriores a la reforma)
# -------------------------------------------------------------------------------------
print("\n(D) Datos para fijar las magnitudes de H1 (grupos anteriores a la reforma):")
pre = ~d["periodo"].astype(str).str.lower().str.startswith("post")
filas_d = []
for g, sub in d[pre].groupby("grupo"):
    s24 = sub[sub["anio_fecha"] == 2024]
    fila = dict(grupo=g, procesos_2024=len(s24))
    if "ind_plazo_ofertas_dias" in sub:
        pl = num(sub["ind_plazo_ofertas_dias"])
        fila["plazo_p5_dias"] = float(np.nanpercentile(pl, 5)) if pl.notna().any() else np.nan
        fila["plazo_2024_con_dato_pct"] = round(100 * float(num(s24["ind_plazo_ofertas_dias"]).notna().mean()), 1) if len(s24) else np.nan
    if "ind_antiguedad_dias" in sub:
        an = num(sub["ind_antiguedad_dias"])
        fila["antig_hasta_30d_pct"] = round(100 * float((an <= 30).mean()), 2)
    if "ind_monto_sobre_capital" in sub:
        mc = num(sub["ind_monto_sobre_capital"])
        fila["monto_capital_p99"] = float(np.nanpercentile(mc, 99)) if mc.notna().any() else np.nan
        fila["capital_2024_con_dato_pct"] = round(100 * float(num(s24["ind_monto_sobre_capital"]).notna().mean()), 1) if len(s24) else np.nan
    if "ind_n_oferentes" in sub:
        fila["dos_o_mas_oferentes_2024_pct"] = round(100 * float((num(s24["ind_n_oferentes"]) >= 2).mean()), 1) if len(s24) else np.nan
    if "precio_reproducido" in sub:
        fila["precio_reproducido_2024_pct"] = round(100 * float(s24["precio_reproducido"].mean()), 1) if len(s24) else np.nan
    filas_d.append(fila)
TAB_D = pd.DataFrame(filas_d).sort_values("procesos_2024", ascending=False)
print(TAB_D.round(2).to_string(index=False))

# -------------------------------------------------------------------------------------
# Veredicto y archivo para el cuaderno 17
# -------------------------------------------------------------------------------------
fallan = [k for k, v in REGLAS.items() if v["coincide_pct"] < 99]
ok_total = (not fallan) and PRECIO.get("ok", False)
print("\nVEREDICTO:", "OK" if ok_total else "REVISAR",
      "" if ok_total else f"(percentiles bajo 99 %: {fallan}; precio ok: {PRECIO.get('ok')})")
salida = dict(fecha=datetime.now().isoformat(timespec="seconds"), veredicto="OK" if ok_total else "REVISAR",
              huella_indicadores=huella(IX), percentiles=REGLAS, concurrencia=CONC, precio=PRECIO,
              magnitudes=TAB_D.replace({np.nan: None}).to_dict(orient="records"))
arch = os.path.join(PREREG, "reproduccion_indicadores.json")
with open(arch, "w", encoding="utf-8") as f:
    json.dump(salida, f, ensure_ascii=False, indent=1, default=float)
TAB_A.to_csv(os.path.join(PREREG, "reproduccion_percentiles_detalle.csv"), index=False)
print(f"Guardado {arch} | huella {huella(arch)} | {time.time() - t0:.0f} s")

In [ ]:
# =====================================================================================
# CELDA F. Firma del preregistro
#   Calcula la huella SHA-256 del preregistro y de todos los archivos que cita, comprueba que
#   coinciden con lo escrito en el documento y deja preregistro_firma.json. No sobrescribe nada.
# =====================================================================================
import os, json, hashlib
from datetime import datetime

SERCOP = r"C:\SERCOP_FINAL"
PREREG = os.path.join(SERCOP, "preregistro")
DOC = os.path.join(PREREG, "preregistro_v1.md")
CORRIDA = os.path.join(SERCOP, "modelos", "20261016_1017_completo_normal_sup")
FIRMANTES = ["Bayron Rafael Urgiles Andrade", "Santiago Calle", "Andrés Canchigña"]

def sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()

# archivo -> huella esperada (16 caracteres) tal como figura en el preregistro
ESPERADAS = {
    os.path.join(SERCOP, "plan_A", "planA_indicadores.parquet"): "dc87f4b8e1282d7f",
    os.path.join(SERCOP, "plan_A", "planA_procesos_modelo.parquet"): "adb1768ebd2dd83a",
    os.path.join(SERCOP, "enmiendas", "enmienda_2026-10-16.json"): "677cfacdb72d7040",
    os.path.join(SERCOP, "enmiendas", "enmienda_2026-10-16_b.json"): "b173395502fd4bfa",
    os.path.join(CORRIDA, "puntajes.parquet"): "eed4fbceb2cc9f3c",
    os.path.join(CORRIDA, "registro_corrida.json"): "979ec56fe8f8935a",
}
REPRO = os.path.join(PREREG, "reproduccion_indicadores.json")

if not os.path.exists(DOC):
    raise SystemExit(f"No encuentro {DOC}. Guarda ahí el archivo preregistro_v1.md.")
texto = open(DOC, encoding="utf-8").read()
if not os.path.exists(REPRO):
    raise SystemExit("Falta reproduccion_indicadores.json: corre primero la celda R.")
veredicto_r = json.load(open(REPRO, encoding="utf-8")).get("veredicto")
if veredicto_r != "OK":
    raise SystemExit("La celda R no terminó en OK: no se firma. Pégame su salida.")

problemas, huellas = [], {}
for path, esperada in ESPERADAS.items():
    if not os.path.exists(path):
        problemas.append(f"no existe {path}")
        continue
    completa = sha(path)
    huellas[os.path.relpath(path, SERCOP)] = completa
    if completa[:16] != esperada:
        problemas.append(f"{os.path.basename(path)}: huella {completa[:16]} distinta de la esperada {esperada}")
    if esperada not in texto:
        problemas.append(f"el preregistro no cita la huella {esperada}")
huellas[os.path.relpath(REPRO, SERCOP)] = sha(REPRO)
if problemas:
    print("NO SE FIRMA. Problemas:")
    for p in problemas:
        print("  -", p)
    raise SystemExit("Pégame estos mensajes.")

firma_path = os.path.join(PREREG, "preregistro_firma.json")
if os.path.exists(firma_path):
    raise SystemExit(f"{firma_path} ya existe: el preregistro ya fue firmado y no se vuelve a firmar.")
firma = dict(documento="preregistro_v1.md", huella_documento=sha(DOC),
             fecha_firma=datetime.now().isoformat(timespec="seconds"), firmantes=FIRMANTES,
             huellas_citadas=huellas, veredicto_celda_R=veredicto_r)
with open(firma_path, "w", encoding="utf-8") as f:
    json.dump(firma, f, ensure_ascii=False, indent=1)
print("PREREGISTRO FIRMADO")
print("  huella del documento:", firma["huella_documento"])
print("  huella de la firma:  ", sha(firma_path))
print("  fecha:", firma["fecha_firma"])
print("Envía estas dos huellas por correo a la tutora (o a ti mismo): la fecha del correo prueba que el")
print("preregistro existía antes de los resultados, sin depender de tu computador.")

In [ ]:
# =====================================================================================
# CELDA 1. Configuración y verificación del preregistro firmado (cuaderno 17: H1 final)
# =====================================================================================
import os, time, json, math, hashlib, platform, warnings, unicodedata
from datetime import datetime
import numpy as np
import pandas as pd
import duckdb
import sklearn
from scipy.stats import spearmanr, norm
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.neural_network import MLPRegressor
from sklearn.exceptions import ConvergenceWarning

# Todo lo que se imprime con say() queda también en resumen_para_chat.txt (carpeta de la corrida)
SALIDA_TXT, CELDA_ACTUAL = {}, "celda 1"
def say(*args):
    texto = " ".join(str(a) for a in args)
    print(texto)
    SALIDA_TXT.setdefault(CELDA_ACTUAL, []).append(texto)

def guardar_txt():
    if "DIR_RUN" in globals():
        with open(os.path.join(DIR_RUN, "resumen_para_chat.txt"), "w", encoding="utf-8") as f:
            for celda, lineas in SALIDA_TXT.items():
                f.write(f"\n##### {celda.upper()} #####\n" + "\n".join(lineas) + "\n")

def empezar(nombre):
    """Marca el inicio de una celda: guarda lo anterior y limpia la salida de esta celda."""
    global CELDA_ACTUAL
    guardar_txt()
    CELDA_ACTUAL = nombre
    SALIDA_TXT[nombre] = []

# --- rutas (cámbialas solo si moviste la carpeta) ---
BASE   = r"C:\SERCOP_FINAL\plan_A"
SALIDA = r"C:\SERCOP_FINAL\modelos"

# --- modo de corrida ---
#   "COMPLETO" : todos los procesos puntuables (desde 2016). Es la corrida que vale para la tesis.
#   "RAPIDO"   : hasta 20.000 procesos por grupo, solo para probar que todo corre.
MODO = "COMPLETO"

# --- cómo ven los modelos cada variable (ver la explicación en el documento) ---
#   "normal_sup" : rareza dentro del grupo pasada a escala normal, contando solo la cola alta (riesgo).
#                  Elegida por el grupo tras el piloto con 2023 (enmienda del 16-oct-2026).
#   "percentil"  : tal como sale de la base; queda solo para comparar.
REPRESENTACION = "normal_sup"

SEMILLA = 20261001                      # semilla global del proyecto

# --- parámetros fijados en el anteproyecto (NO se cambian mirando resultados) ---
IF_ARBOLES    = 100                     # Isolation Forest: árboles (Liu et al., 2008)
IF_SUBMUESTRA = 256                     # Isolation Forest: tamaño de submuestra por árbol
LOF_VECINOS   = 20                      # Local Outlier Factor: vecinos (Breunig et al., 2000)
CORTE_TOP     = 0.01                    # lista priorizada: 1 % superior de cada grupo
MIN_PATRON    = 1000                    # patrón de disponibilidad con modelo propio

# --- enmienda del 16-oct-2026 (decidida antes de evaluar H1, H2 y H3) ---
ANIO_INICIO   = 2016                    # 2015 solo sirve de historial: no se puntúa

# --- parámetros de cómputo (no cambian la definición del modelo, solo el costo) ---
MIN_GRUPO        = 100                  # grupos más chicos se reportan pero no se modelan
REF_MAX          = 50_000               # muestra de referencia máxima por modelo
AE_MAX_ENTRENO   = 50_000               # filas máximas para entrenar el autoencoder
HUERFANO_MAX     = 10_000               # lo mismo, para los modelos de patrones pequeños
RAPIDO_MAX_GRUPO = 20_000               # procesos por grupo en modo RAPIDO

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M") + "_h1final"
DIR_RUN = os.path.join(SALIDA, RUN_ID)
os.makedirs(DIR_RUN, exist_ok=True)
T_INICIO = time.time()
ALERTAS = []                            # aquí se acumulan los avisos que se resumen en la celda 9

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")
warnings.filterwarnings("ignore", category=RuntimeWarning, message="Mean of empty slice")
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)

say("Corrida:", RUN_ID)
say("Carpeta de salida:", DIR_RUN)
say(f"Python {platform.python_version()} | numpy {np.__version__} | pandas {pd.__version__} | "
      f"scikit-learn {sklearn.__version__} | duckdb {duckdb.__version__} | núcleos {os.cpu_count()}")
say(f"Parámetros fijados: IF {IF_ARBOLES} árboles x {IF_SUBMUESTRA} | LOF k={LOF_VECINOS} | "
      f"corte {CORTE_TOP:.0%} | patrón mínimo {MIN_PATRON} | representación '{REPRESENTACION}' | "
      f"se puntúa desde {ANIO_INICIO}")
if MODO != "COMPLETO" or REPRESENTACION != "normal_sup":
    raise SystemExit("El cuaderno 17 solo corre en modo COMPLETO con normal_sup, como dice el preregistro.")

# --- el cuaderno solo corre si el preregistro está firmado y nada cambió desde la firma ---
SERCOP = os.path.dirname(BASE.rstrip("\\/"))
PREREG = os.path.join(SERCOP, "preregistro")

def sha_completa(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()

firma_path = os.path.join(PREREG, "preregistro_firma.json")
if not os.path.exists(firma_path):
    raise SystemExit("El preregistro no está firmado: corre antes la celda F.")
FIRMA = json.load(open(firma_path, encoding="utf-8"))
if sha_completa(os.path.join(PREREG, FIRMA["documento"])) != FIRMA["huella_documento"]:
    raise SystemExit("El preregistro cambió después de firmarlo: no se corre H1.")
for rel, h in FIRMA["huellas_citadas"].items():
    if sha_completa(os.path.join(SERCOP, rel)) != h:
        raise SystemExit(f"{rel} cambió después de la firma: no se corre H1.")
say(f"Preregistro verificado: huella {FIRMA['huella_documento'][:16]}, firmado el {FIRMA['fecha_firma']}")

In [ ]:
# =====================================================================================
# CELDA 2. Carga de la base entregada y armado de los grupos comparables
#          (NO se carga planA_validacion_sri: la marca del SRI se usa una sola vez, al final)
# =====================================================================================
empezar("celda 2")
def ruta(nombre):
    return os.path.join(BASE, nombre).replace("\\", "/")

def huella(path, n=16):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()[:n]

def a_float(s):
    """Convierte booleanos, enteros o nulos a float (nulo -> NaN) sin errores de tipo."""
    try:
        return pd.Series(pd.array(s, dtype="Float64").to_numpy(dtype="float64", na_value=np.nan), index=s.index)
    except Exception:
        return pd.to_numeric(s.astype("object"), errors="coerce").astype("float64")

def guardar_parquet(tabla, nombre):
    """Guarda en la carpeta de la corrida; si pandas no tiene motor de parquet, usa DuckDB."""
    destino = os.path.join(DIR_RUN, nombre)
    try:
        tabla.to_parquet(destino)
    except ImportError:
        c = duckdb.connect()
        c.register("t_guardar", tabla.reset_index())
        c.execute(f"COPY t_guardar TO '{destino.replace(chr(92), '/')}' (FORMAT PARQUET)")
        c.close()

def sin_tildes(texto):
    t = unicodedata.normalize("NFKD", str(texto))
    return "".join(c for c in t if not unicodedata.combining(c)).lower().strip()

t_celda = time.time()
IX = ruta("planA_indicadores.parquet")
PM = ruta("planA_procesos_modelo.parquet")
for p in (IX, PM):
    if not os.path.exists(p):
        raise SystemExit(f"No encuentro {p}. Revisa BASE en la celda 1.")
HUELLA_IX, HUELLA_PM = huella(IX), huella(PM)

con = duckdb.connect()
ix = con.execute(f"SELECT * FROM read_parquet('{IX}')").df()
cols_pm = [r[0] for r in con.execute(f"DESCRIBE SELECT * FROM read_parquet('{PM}')").fetchall()]
pedir = ["ocid"] + [c for c in ["familia", "fecha_inicio_tender", "monto_final", "entidad_ruc", "proveedor_ruc_final"]
                    if c in cols_pm and c not in ix.columns]
pm = con.execute(f"SELECT {', '.join(pedir)} FROM read_parquet('{PM}')").df()
say(f"planA_indicadores: {ix.shape[0]:,} filas x {ix.shape[1]} columnas | huella {HUELLA_IX}")
say(f"planA_procesos_modelo: {pm.shape[0]:,} filas; se toman {pedir[1:]} | huella {HUELLA_PM}")
say("\nColumnas de planA_indicadores:")
say(", ".join(ix.columns))

df = ix.merge(pm, on="ocid", how="left")
if len(df) != len(ix) or df["ocid"].duplicated().any():
    raise SystemExit("La unión por ocid cambió el número de filas: revisar antes de seguir.")
say(f"\nProcesos de indicadores sin pareja en procesos_modelo: {(~ix['ocid'].isin(pm['ocid'])).sum():,}")
if "familia" not in df.columns:
    raise SystemExit("No encuentro la columna 'familia' en ninguno de los dos archivos.")

# fecha del proceso = inicio del período de ofertas
col_fecha = next((c for c in ["fecha_inicio_tender", "fecha_proceso"] if c in df.columns), None)
if col_fecha is None:
    raise SystemExit("No encuentro la fecha del proceso (fecha_inicio_tender).")
fecha = pd.to_datetime(df[col_fecha], errors="coerce")
try:
    if fecha.dt.tz is not None:
        fecha = fecha.dt.tz_convert("America/Guayaquil").dt.tz_localize(None)
except AttributeError:
    pass
say(f"Fecha del proceso tomada de '{col_fecha}'; procesos sin fecha: {int(fecha.isna().sum()):,}")
df["anio_proc"] = fecha.dt.year
df["periodo_calc"] = np.where(fecha >= pd.Timestamp("2025-10-07"), "post", "pre")
if "periodo" in df.columns:
    coinc = (df["periodo"].astype(str).str.lower().str.startswith("post") == (df["periodo_calc"] == "post")).mean()
    say(f"Coincidencia entre 'periodo' de la base y el corte por fecha: {coinc:.4%}")
df["grupo"] = df["familia"].astype(str) + " | " + df["periodo_calc"]
if "monto_final" not in df.columns:
    df["monto_final"] = np.nan
df["monto_final"] = a_float(df["monto_final"])

# Control de calibración sobre la base completa (antes de quitar 2015 y de muestrear):
# cada percentil de la base debe tener media 0,500 dentro de su grupo
say("\nControl de calibración de la base (media de cada percentil dentro de su grupo; debe dar 0,500):")
filas = []
for c in ["pct_precio_alto", "pct_monto_capital", "pct_proveedor_nuevo", "pct_plazo_corto", "pct_firma_rapida",
          "pct_fraccionamiento_pos", "pct_cuota_pos", "pct_recurrencia_pos"]:
    if c in df.columns:
        s_ = a_float(df[c])
        g_ = s_.groupby(df["grupo"])
        m = g_.mean()[g_.count() >= 200]                 # solo grupos con al menos 200 valores
        filas.append(dict(columna=c, media_min=m.min(), media_max=m.max(),
                          grupo_mas_lejos=(m - 0.5).abs().idxmax() if m.notna().any() else "-"))
cal = pd.DataFrame(filas)
say(cal.round(4).to_string(index=False))
lejos = cal[(cal["media_min"] < 0.495) | (cal["media_max"] > 0.505)]
if len(lejos):
    ALERTAS.append("Percentiles de la base con media lejos de 0,5 en algún grupo: " + ", ".join(lejos["columna"]))

# Debutante con ventana de 12 meses (enmienda): sin procesos del proveedor en los 365 días anteriores.
# Sale de la misma columna de historial que usan cuota y recurrencia (verificada en la celda 0).
if "ind_procesos_proveedor_12m" not in df.columns:
    raise SystemExit("No encuentro ind_procesos_proveedor_12m: corre primero la celda 0 y pégame su salida.")
n12 = a_float(df["ind_procesos_proveedor_12m"])
df["ind_proveedor_debutante_365"] = np.where(n12.isna(), np.nan, (n12 == 0).astype(float))

# 2015 queda solo como historial (enmienda): se retira de lo que se puntúa
DF_TODO = df.copy()     # base completa (con 2015): de aquí salen las distribuciones del recálculo
quitados = df.loc[df["anio_proc"] < ANIO_INICIO].groupby("grupo").size()
df = df.loc[df["anio_proc"] >= ANIO_INICIO].reset_index(drop=True)
say(f"\nProcesos anteriores a {ANIO_INICIO} retirados (siguen sirviendo de historial): {int(quitados.sum()):,}")
say(f"Procesos puntuables: {len(df):,}")

TAM_REAL = df.groupby("grupo").size()            # tamaño de cada grupo puntuable en la base completa
if MODO == "RAPIDO":
    df = (df.sample(frac=1.0, random_state=SEMILLA)
            .groupby("grupo", sort=False).head(RAPIDO_MAX_GRUPO)
            .sort_values("ocid").reset_index(drop=True))
    say(f"\nModo RAPIDO: se trabaja con {len(df):,} procesos (hasta {RAPIDO_MAX_GRUPO:,} por grupo).")

tab = (df.groupby("grupo").agg(procesos=("ocid", "size"),
                               de_2023=("anio_proc", lambda s: int((s == 2023).sum())),
                               anio_min=("anio_proc", "min"), anio_max=("anio_proc", "max"),
                               millones=("monto_final", lambda s: round(s.sum() / 1e6, 1))))
tab.insert(0, "en_la_base", TAM_REAL)
tab.insert(1, f"quitados_antes_{ANIO_INICIO}", quitados.reindex(tab.index).fillna(0).astype(int))
tab = tab.sort_values("en_la_base", ascending=False)
say("\nGrupos comparables (familia | período):")
say(tab.to_string())
say("\nRecordatorio: planA_validacion_sri NO se cargó. Ninguna cifra de esta corrida usa la marca del SRI.")
say(f"[celda 2: {time.time() - t_celda:.0f} s]")

In [ ]:
# =====================================================================================
# CELDA 3. Codificación de las variables del modelo (sin imputar constantes)
#   rango      -> percentil del grupo (ya viene en la base); si falta el dato queda vacío (NaN)
#   rango_pos  -> percentil entre los procesos donde el fenómeno ocurre; 0 cuando no ocurre
#                 (cero real: no hubo historial o no hubo recompra), nunca un relleno
#   binaria    -> 0/1; vacío si falta
#   dif        -> diferencia con la mediana de oferentes del grupo, llevada a 0-1 dentro del grupo;
#                 1 cuando hubo un solo oferente (el extremo real de la escala)
# =====================================================================================
empezar("celda 3")
t_celda = time.time()

def pct_mitad_salto(s, grupos):
    """Percentil a mitad de salto dentro de cada grupo (los vacíos quedan vacíos)."""
    r = s.groupby(grupos).rank(method="average")
    n = s.notna().groupby(grupos).transform("sum")
    return (r - 0.5) / n

VARIABLES = {
    # nombre             columna en la base                  tipo         entra a la línea base
    "precio_alto":       ("pct_precio_alto",                 "rango",     True),
    "monto_capital":     ("pct_monto_capital",               "rango",     True),
    "proveedor_nuevo":   ("pct_proveedor_nuevo",             "rango",     True),
    "plazo_corto":       ("pct_plazo_corto",                 "rango",     True),
    "firma_rapida":      ("pct_firma_rapida",                "rango",     True),
    "recompra_30d":      ("pct_fraccionamiento_pos",         "rango_pos", True),
    "cuota_entidad":     ("pct_cuota_pos",                   "rango_pos", True),
    "recurrencia_par":   ("pct_recurrencia_pos",             "rango_pos", True),
    "oferente_unico":    ("ind_oferente_unico",              "binaria",   True),
    "poca_concurrencia": ("dif_concurrencia_grupo",          "dif",       True),
    # primera_vez_ent sale de los modelos: con la ventana de 12 meses coincide exactamente con
    # "recurrencia = 0", así que no aporta información nueva (enmienda del 16-oct-2026)
    "debutante":         ("ind_proveedor_debutante_365",     "binaria",   True),
}
# columnas de origen de las variables en dos partes: la que dice si el fenómeno ocurre (> 0) y la
# que mide su tamaño (solo se usa si hubiera que reconstruir el percentil)
ORIGEN_POS = {"recompra_30d":    ["ind_fraccionamiento_marca", "ind_fraccionamiento_monto_30d"],
              "cuota_entidad":   ["ind_cuota_prov_en_entidad_12m"],
              "recurrencia_par": ["ind_recurrencia_par_12m"]}
MAGNITUD_POS = {"recompra_30d":    ["ind_fraccionamiento_monto_30d"],
                "cuota_entidad":   ["ind_cuota_prov_en_entidad_12m"],
                "recurrencia_par": ["ind_recurrencia_par_12m"]}

def primera(cols):
    return next((c for c in cols if c in df.columns), None)

# Si una columna no existe con el nombre esperado, se reconstruye desde su origen (y se avisa)
RECONSTRUIDAS = []
for v, origenes in MAGNITUD_POS.items():
    col, origen = VARIABLES[v][0], primera(origenes)
    if col not in df.columns and origen is not None:
        mag = a_float(df[origen])
        df[col] = pct_mitad_salto(mag.where(mag > 0), df["grupo"])
        RECONSTRUIDAS.append(f"{v} desde {origen}")
col_nof = primera(["ind_n_oferentes", "n_oferentes"])
if col_nof is not None:
    nof = a_float(df[col_nof])
    if "ind_oferente_unico" not in df.columns:
        df["ind_oferente_unico"] = np.where(nof.isna(), np.nan, (nof == 1).astype(float))
        RECONSTRUIDAS.append(f"oferente_unico desde {col_nof}")
    if "dif_concurrencia_grupo" not in df.columns:
        df["dif_concurrencia_grupo"] = (nof.groupby(df["grupo"]).transform("median") - nof).where(nof >= 2)
        RECONSTRUIDAS.append(f"poca_concurrencia desde {col_nof}")

X = pd.DataFrame(index=df.index)
say("Variables del modelo y columna de origen:")
for v, (col, tipo, _) in list(VARIABLES.items()):
    if col not in df.columns:
        say(f"  {v:<18} NO ENCONTRADA ({col}) -> queda fuera")
        ALERTAS.append(f"Variable {v} no encontrada en la base (columna {col}).")
        VARIABLES.pop(v)
        continue
    s = a_float(df[col])
    nota = ""
    if tipo == "rango_pos":
        origen = primera(ORIGEN_POS[v])
        if origen is not None:
            o = a_float(df[origen])
            ocurre, no_ocurre = o > 0, o <= 0
            mal = int((s.isna() & ocurre).sum() + ((s > 0) & no_ocurre).sum())
            nota = f"control de ceros con {origen}: {mal:,} filas incoherentes" + ("  <- REVISAR" if mal else " (ok)")
            if mal:
                ALERTAS.append(f"{v}: {mal:,} filas donde el percentil y su origen no coinciden.")
            s = s.where(~(no_ocurre & s.isna()), 0.0)
        else:
            nota = "sin columna de origen: vacío se toma como 'no ocurre'"
            s = s.fillna(0.0)
    X[v] = s
    say(f"  {v:<18} {col:<32} {tipo:<10} cobertura {100 * s.notna().mean():5.1f} %   {nota}")
if RECONSTRUIDAS:
    say("Reconstruidas porque la columna esperada no existe:", "; ".join(RECONSTRUIDAS))
    ALERTAS.append("Variables reconstruidas desde su origen: " + "; ".join(RECONSTRUIDAS))
if X.shape[1] < 6:
    raise SystemExit("Faltan demasiadas variables. Pega en el chat la lista de columnas de la celda 2.")

# poca_concurrencia: escala 0-1 dentro de cada grupo, 1 = un solo oferente
if "poca_concurrencia" in X.columns:
    pc = X["poca_concurrencia"].copy()
    for g, idx in df.groupby("grupo").groups.items():
        vals = pc.loc[idx].dropna()
        if len(vals) >= 30:
            lo, hi = np.nanpercentile(vals, 1), np.nanpercentile(vals, 99)
            if hi > lo:
                pc.loc[idx] = ((pc.loc[idx].clip(lo, hi) - lo) / (hi - lo))
            else:
                pc.loc[idx] = np.where(pc.loc[idx].notna(), 0.5, np.nan)
        else:
            pc.loc[idx] = np.where(pc.loc[idx].notna(), 0.5, np.nan)
    if "oferente_unico" in X.columns:
        pc = pc.where(~(X["oferente_unico"] == 1), 1.0)
        pc = pc.where(~X["oferente_unico"].isna(), np.nan)
    X["poca_concurrencia"] = pc

ORIENTADAS = [v for v, (_, _, base) in VARIABLES.items() if base]
TIPO = {v: t for v, (_, t, _) in VARIABLES.items()}
CORTO = {"precio_alto": "precio", "monto_capital": "capital", "proveedor_nuevo": "prov_nuevo",
         "plazo_corto": "plazo", "firma_rapida": "firma", "recompra_30d": "recompra",
         "cuota_entidad": "cuota", "recurrencia_par": "recurr", "oferente_unico": "of_unico",
         "poca_concurrencia": "poca_conc", "primera_vez_ent": "primera_vez", "debutante": "debutante"}

# control de orientación (alto = anómalo); la calibración ya se revisó en la celda 2
def control_orientacion(var, origen, signo):
    if var in X.columns and origen in df.columns:
        a, b = X[var], a_float(df[origen])
        ok = a.notna() & b.notna() & ((a > 0) if TIPO[var] == "rango_pos" else True)
        if ok.sum() > 100:
            r = spearmanr(a[ok], b[ok]).correlation
            estado = "ok" if np.sign(r) == signo else "REVISAR ORIENTACIÓN"
            if estado != "ok":
                ALERTAS.append(f"Orientación dudosa: {var} frente a {origen} (rho {r:+.2f}).")
            say(f"  {var:<18} vs {origen:<32} rho = {r:+.3f}  (esperado {'+' if signo > 0 else '-'})  {estado}")
say("\nControl de orientación (alto = más anómalo):")
control_orientacion("precio_alto", "ind_desv_precio_unit_z", +1)
control_orientacion("monto_capital", "ind_monto_sobre_capital", +1)
control_orientacion("proveedor_nuevo", "ind_antiguedad_dias", -1)
control_orientacion("plazo_corto", "ind_plazo_ofertas_dias", -1)
control_orientacion("firma_rapida", "ind_plazo_adj_firma_dias", -1)
control_orientacion("recompra_30d", "ind_fraccionamiento_monto_30d", +1)
control_orientacion("cuota_entidad", "ind_cuota_prov_en_entidad_12m", +1)
control_orientacion("recurrencia_par", "ind_recurrencia_par_12m", +1)
if col_nof is not None:
    control_orientacion("poca_concurrencia", col_nof, -1)

cob = (X.notna().groupby(df["grupo"]).mean() * 100).round(1)
cob.insert(0, "procesos", df.groupby("grupo").size())
say("\nCobertura de cada variable por grupo (% de procesos con dato):")
say(cob.sort_values("procesos", ascending=False).rename(columns=CORTO).to_string())

guardar_parquet(X, "X_codificado.parquet")
guardar_parquet(df[["ocid", "grupo", "familia", "periodo_calc", "anio_proc", "monto_final"]
                   + [c for c in ["entidad_ruc", "proveedor_ruc_final"] if c in df.columns]], "llaves.parquet")
say(f"[celda 3: {time.time() - t_celda:.0f} s]")

In [ ]:
# =====================================================================================
# CELDA 4. Patrones de disponibilidad: qué variables usa cada modelo en cada grupo
#   - Se retiran del grupo las variables casi ausentes (<1 %) o constantes (>= 99,5 % un valor).
#   - Cada patrón con >= MIN_PATRON procesos tiene modelo propio ("grande").
#   - Un patrón menor se puntúa con el modelo grande que usa solo variables que él sí tiene;
#     si no hay ninguno, recibe un modelo propio ajustado sin la variable que le falta ("pequeño").
# =====================================================================================
empezar("celda 4")
def variables_del_grupo(Xg):
    usar, fuera = [], []
    for v in Xg.columns:
        s = Xg[v].dropna()
        if len(s) < max(30, 0.01 * len(Xg)):
            fuera.append(f"{v} (sin dato)")
        elif s.value_counts(normalize=True).iloc[0] >= 0.995:
            fuera.append(f"{v} (constante)")
        else:
            usar.append(v)
    return usar, fuera

def plan_de_modelos(Xg, usar):
    disp = Xg[usar].notna().to_numpy()
    codigo = (disp * (1 << np.arange(len(usar)))).sum(axis=1)
    texto = {c: "".join("1" if (c >> j) & 1 else "0" for j in range(len(usar))) for c in np.unique(codigo)}
    clave = pd.Series([texto[c] for c in codigo], index=Xg.index)
    conteo = clave.value_counts()
    grandes = [k for k, n in conteo.items() if n >= MIN_PATRON] or [conteo.index[0]]
    def vars_de(k): return [u for u, b in zip(usar, k) if b == "1"]
    asign = {}
    for k in conteo.index:
        if k in grandes:
            asign[k] = k
            continue
        mias = set(vars_de(k))
        cand = [g for g in grandes if set(vars_de(g)) <= mias]
        asign[k] = max(cand, key=lambda g: (len(vars_de(g)), conteo[g])) if cand else k
    modelos = sorted(set(asign.values()), key=lambda k: -conteo.get(k, 0))
    return clave, conteo, asign, {m: vars_de(m) for m in modelos}, set(grandes)

PLAN, NO_MODELADOS, filas = {}, {}, []
for g in df.groupby("grupo").size().sort_values(ascending=False).index:
    idx = df.index[df["grupo"] == g]
    Xg = X.loc[idx]
    usar, fuera = variables_del_grupo(Xg)
    if len(idx) < MIN_GRUPO or len(usar) < 2:
        NO_MODELADOS[g] = f"{len(idx)} procesos, {len(usar)} variables útiles"
        continue
    clave, conteo, asign, vars_modelo, grandes = plan_de_modelos(Xg, usar)
    PLAN[g] = dict(idx=idx, usar=usar, clave=clave, asign=asign, vars_modelo=vars_modelo, grandes=grandes)
    modelo_de = clave.map(asign)
    filas.append(dict(grupo=g, procesos=len(idx), variables=len(usar), patrones=len(conteo),
                      modelos_grandes=sum(m in grandes for m in vars_modelo),
                      modelos_pequenos=sum(m not in grandes for m in vars_modelo),
                      pct_en_modelo_principal=round(100 * (modelo_de == conteo.index[0]).mean(), 1),
                      pct_en_modelos_pequenos=round(100 * (~modelo_de.isin(grandes)).mean(), 2),
                      vars_modelo_principal=conteo.index[0].count("1"),
                      fuera=", ".join(fuera)))
say(pd.DataFrame(filas).to_string(index=False))
if NO_MODELADOS:
    say("\nGrupos que no se modelan:", NO_MODELADOS)

In [ ]:
# =====================================================================================
# CELDA 5. Funciones de los modelos (las mismas del cuaderno 16 versión 3; aquí no se ajusta nada)
# =====================================================================================
empezar("celda 5")
def pct_vs_ref(ref, x):
    """Percentil a mitad de salto de cada valor de x frente a la distribución de referencia ref."""
    ref = np.sort(np.asarray(ref, dtype=float))
    lo = np.searchsorted(ref, x, side="left")
    hi = np.searchsorted(ref, x, side="right")
    return (lo + 0.5 * (hi - lo)) / len(ref)

def ajustar_un_modelo(Xtr, Xsc, semilla, ref_max, ae_max):
    """Entrena los tres detectores con Xtr (procesos que tienen todas las variables del modelo)
    y puntúa Xsc. Devuelve percentiles (alto = más anómalo) frente a una referencia tomada de Xtr."""
    rng = np.random.default_rng(semilla)
    n, d = Xtr.shape
    R = Xtr[rng.choice(n, size=min(n, ref_max), replace=False)]
    out, tiempos = {}, {}
    # Isolation Forest
    t = time.time()
    iso = IsolationForest(n_estimators=IF_ARBOLES, max_samples=min(IF_SUBMUESTRA, n),
                          random_state=semilla, n_jobs=-1).fit(Xtr)
    out["if"] = pct_vs_ref(-iso.score_samples(R), -iso.score_samples(Xsc))
    tiempos["if"] = time.time() - t
    # Local Outlier Factor (modo novedad). Se ajusta sobre las filas distintas de la referencia.
    # Un proceso idéntico a una fila de la referencia recibe el LOF de esa fila, calculado sin
    # contarse a sí misma; así todos los procesos se miden con la misma regla.
    t = time.time()
    U, inv = np.unique(R, axis=0, return_inverse=True)
    inv = np.asarray(inv).ravel()
    if len(U) > LOF_VECINOS + 5:
        lof = LocalOutlierFactor(n_neighbors=LOF_VECINOS, novelty=True, n_jobs=-1).fit(U)
        lof_u = -lof.negative_outlier_factor_
        _, inv_all = np.unique(np.vstack([U, Xsc]), axis=0, return_inverse=True)
        inv_all = np.asarray(inv_all).ravel()
        pos = np.full(inv_all.max() + 1, -1)
        pos[inv_all[:len(U)]] = np.arange(len(U))
        m = pos[inv_all[len(U):]]
        s = np.empty(len(Xsc))
        s[m >= 0] = lof_u[m[m >= 0]]
        if (m < 0).any():
            s[m < 0] = -lof.score_samples(Xsc[m < 0])
        out["lof"] = pct_vs_ref(lof_u[inv], s)
    else:
        out["lof"] = np.full(len(Xsc), np.nan)
    tiempos["lof"] = time.time() - t
    # Autoencoder: red simétrica con capa intermedia de la mitad de las variables
    t = time.time()
    if d >= 3:
        tr_i = rng.choice(n, size=min(n, ae_max), replace=False)
        ae = MLPRegressor(hidden_layer_sizes=(max(2, d // 2),), activation="relu", solver="adam",
                          alpha=1e-4, batch_size=256, learning_rate_init=1e-3, max_iter=200,
                          early_stopping=True, validation_fraction=0.1, n_iter_no_change=10,
                          random_state=semilla)
        ae.fit(Xtr[tr_i], Xtr[tr_i])
        err_ref = ((ae.predict(R) - R) ** 2).mean(axis=1)
        err_sc = ((ae.predict(Xsc) - Xsc) ** 2).mean(axis=1)
        out["ae"] = pct_vs_ref(err_ref, err_sc)
    else:
        out["ae"] = np.full(len(Xsc), np.nan)
    tiempos["ae"] = time.time() - t
    return out, tiempos

def top_k(valores, frac, semilla):
    """Marca el frac superior (exactamente k procesos con puntaje; empates al azar con semilla fija)."""
    v = np.asarray(valores, dtype=float)
    fin = np.isfinite(v)
    marca = np.zeros(len(v), dtype=bool)
    if fin.sum() == 0:
        return marca
    k = max(1, int(math.ceil(frac * fin.sum())))
    desempate = np.random.default_rng(semilla).random(len(v))
    orden = np.lexsort((desempate, -np.where(fin, v, -np.inf)))
    marca[orden[:k]] = True
    return marca

def rareza(Xg_df):
    """Percentil a mitad de salto de cada variable dentro del conjunto (alto = valor más alto)."""
    return (Xg_df.rank(method="average") - 0.5) / Xg_df.notna().sum()

def entrada_modelo(Xg_df):
    """Lo que reciben los modelos. 'percentil': los valores tal cual. 'normal_sup': la rareza de cada
    valor dentro del grupo en escala normal, con la mitad baja (sin riesgo) llevada a 0."""
    if REPRESENTACION == "normal_sup":
        z = norm.ppf(rareza(Xg_df).clip(1e-9, 1 - 1e-9).to_numpy(dtype=float))
        return pd.DataFrame(np.maximum(z, 0.0), index=Xg_df.index, columns=Xg_df.columns).where(Xg_df.notna())
    return Xg_df

def puntuar_grupo(plan, Xg_df, semilla, verbose=False):
    """Ajusta todos los modelos de un grupo. Devuelve percentiles por proceso, tiempos y rareza."""
    res = pd.DataFrame(index=Xg_df.index)
    Xm = entrada_modelo(Xg_df[plan["usar"]])
    for m in ["if", "lof", "ae"]:
        res[m] = np.nan
    res["modelo"] = ""
    t_mod = {"if": 0.0, "lof": 0.0, "ae": 0.0}
    clave = plan["clave"].loc[Xg_df.index]
    modelo_de = clave.map(plan["asign"])
    for j, (mk, vs) in enumerate(plan["vars_modelo"].items(), start=1):
        tm = time.time()
        destino = modelo_de.index[modelo_de == mk]
        if len(destino) == 0 or len(vs) == 0:
            continue
        tr_mask = Xg_df[vs].notna().all(axis=1)
        if tr_mask.sum() < 30:
            continue
        tope = (REF_MAX, AE_MAX_ENTRENO) if mk in plan["grandes"] else (HUERFANO_MAX, HUERFANO_MAX)
        out, tt = ajustar_un_modelo(Xm.loc[tr_mask, vs].to_numpy(dtype=float),
                                    Xm.loc[destino, vs].to_numpy(dtype=float), semilla, *tope)
        for m in out:
            res.loc[destino, m] = out[m]
            t_mod[m] += tt[m]
        res.loc[destino, "modelo"] = mk
        if verbose:
            say(f"      modelo {j}/{len(plan['vars_modelo'])}: {len(vs)} variables, entrena con "
                  f"{int(tr_mask.sum()):,}, puntúa {len(destino):,}  ({time.time() - tm:.0f} s)")
    res["ensamble"] = res[["if", "lof", "ae"]].mean(axis=1, skipna=True)
    rz = rareza(Xg_df[plan["usar"]])
    orient = [v for v in ORIENTADAS if v in plan["usar"]]
    res["base"] = rz[orient].mean(axis=1, skipna=True) if orient else np.nan
    res["n_vars_disp"] = Xg_df[plan["usar"]].notna().sum(axis=1)
    return res, t_mod, rz

def explicar(rz, cols, n=3):
    """Para cada fila, las n variables con valor más alto dentro del grupo."""
    a = rz[cols].to_numpy(dtype=float)
    a = np.where(np.isfinite(a), a, -1.0)
    orden = np.argsort(-a, axis=1)[:, :n]
    return [" | ".join(f"{cols[j]} {a[i, j]:.2f}" for j in orden[i] if a[i, j] >= 0) for i in range(len(a))]

def marcar_tops(res, semilla):
    for m in ["ensamble", "if", "lof", "ae", "base"]:
        res[f"top_{m}"] = top_k(res[m], CORTE_TOP, semilla)
    res["consenso"] = res[["top_if", "top_lof", "top_ae"]].sum(axis=1) >= 2
    return res
say("Funciones de los modelos cargadas.")

In [ ]:
# =====================================================================================
# CELDA 6. Recetas de inyección: cómo se altera cada variable de origen y cómo se recalcula
#          su indicador con la misma regla de la base (la que verificó la celda R)
# =====================================================================================
empezar("celda 6")
t_celda = time.time()
REPRO = json.load(open(os.path.join(PREREG, "reproduccion_indicadores.json"), encoding="utf-8"))
if REPRO.get("veredicto") != "OK" or REPRO.get("huella_indicadores") != HUELLA_IX:
    raise SystemExit("La reproducción de la celda R no es OK o es de otra versión de la base.")
REGLAS = REPRO["percentiles"]

def num(s):
    return a_float(s)

def clave_grupo(tabla, agrupacion):
    if agrupacion == "familia|periodo":
        return tabla["familia"].astype(str) + " | " + tabla["periodo"].astype(str)
    return tabla["familia"].astype(str)

def valores_orientados(tabla, regla):
    x = num(tabla[regla["fuente"]])
    if regla["subconjunto"] == "positivos":
        x = x.where(x > 0)
    return regla["signo"] * x

# distribución original de cada fuente en la base completa (con 2015, como se construyó la base)
ORDENADOS = {}
for pct, regla in REGLAS.items():
    v = valores_orientados(DF_TODO, regla)
    k = clave_grupo(DF_TODO, regla["agrupacion"])
    ORDENADOS[pct] = {g: np.sort(s.dropna().to_numpy()) for g, s in v.groupby(k)}

def recalcular(pct, filas, nuevos_origen):
    """Nuevo percentil de las filas `filas` de df si su variable de origen pasa a `nuevos_origen`.
    Se compara con la distribución original del grupo, quitando el valor original del propio proceso."""
    regla = REGLAS[pct]
    sub = df.loc[filas]
    claves = clave_grupo(sub, regla["agrupacion"]).to_numpy()
    orig = valores_orientados(sub, regla).to_numpy()
    nuevo = regla["signo"] * np.asarray(nuevos_origen, dtype=float)
    salida = np.full(len(filas), np.nan)
    for g in np.unique(claves):
        m = claves == g
        arr = ORDENADOS[pct].get(g, np.array([]))
        v, o = nuevo[m], orig[m]
        debajo = np.searchsorted(arr, v, side="left").astype(float)
        iguales = np.searchsorted(arr, v, side="right") - debajo
        propio = ~np.isnan(o)                         # el valor original está dentro de arr
        debajo -= (propio & (o < v))
        iguales -= (propio & (o == v))
        n = len(arr) + (~propio).astype(float)        # si no estaba, el proceso entra al conjunto
        r_med = debajo + (iguales + 2) / 2            # rango medio del bloque de empates, con el propio
        r_min, r_max = debajo + 1, debajo + iguales + 1
        f = regla["formula"]
        if f == "mitad_de_salto":
            p = (r_med - 0.5) / n
        elif f == "rango_medio/n":
            p = r_med / n
        elif f.startswith("acumulada"):
            p = r_max / n
        else:
            p = (r_min - 1) / (n - 1)
        salida[m] = p
    return salida

# control: con el valor ORIGINAL, el recálculo debe devolver exactamente el percentil de la base
say("Control del recálculo (valor original -> percentil de la base, % idéntico en 2.000 procesos al azar):")
rng_c = np.random.default_rng(SEMILLA)
for pct, regla in REGLAS.items():
    con_dato = df.index[valores_orientados(df, regla).notna() & num(df[pct]).notna()]
    if len(con_dato) == 0:
        continue
    filas = pd.Index(rng_c.choice(con_dato, size=min(2000, len(con_dato)), replace=False))
    p = recalcular(pct, filas, num(df.loc[filas, regla["fuente"]]).to_numpy())
    igual = 100 * np.mean(np.abs(p - num(df.loc[filas, pct]).to_numpy()) < 1e-6)
    say(f"  {pct:<26} {igual:6.2f} %")
    if igual < 99:
        raise SystemExit(f"El recálculo de {pct} no reproduce la base: pégame esta salida.")

# precio: grupo de referencia, mediana y escala de cada proceso (regla de la celda R)
PRECIO = REPRO["precio"]
t = DF_TODO.copy()
pu = num(t["precio_unit"])
t["lp"] = np.log(pu.where(pu > 0))
t["z"] = num(t["ind_desv_precio_unit_z"])
c5 = t["cpc5"].astype(str).str.replace(r"\.0$", "", regex=True).str.replace(r"\D", "", regex=True)
t["cpc5_s"] = c5.where(c5 != "", np.nan).str.zfill(5)
t["cpc3_s"] = t["cpc5_s"].str[:3]
t["unidad_s"] = t["unidad"].astype(str).str.strip().str.upper() if "unidad" in t.columns else "-"
t["unidad_raw"] = t["unidad"] if "unidad" in t.columns else "-"
t["familia_s"] = t["familia"].astype(str)
t["anio_fecha"] = pd.to_datetime(t["fecha_proceso"], errors="coerce").dt.year
niv = t["ind_precio_nivel"].astype("object")
t["m_ref"], t["mad_ref"] = np.nan, np.nan
for etiqueta, cfg in PRECIO["niveles"].items():
    es = niv.astype(str) == etiqueta
    keys, alcance = cfg["claves"], cfg["alcance"]
    m = t["lp"].notna() & (es if alcance == "mismo_nivel" else True)
    sub = t.loc[m, keys + ["lp"]]
    med = sub.groupby(keys, dropna=False)["lp"].transform("median")
    mad = (sub["lp"] - med).abs().groupby([sub[k] for k in keys], dropna=False).transform("median")
    t.loc[es, "m_ref"] = med.reindex(t.index)[es]
    t.loc[es, "mad_ref"] = mad.reindex(t.index)[es]
ESCALA = PRECIO["c_mad"] * t["mad_ref"]
z_hat = (t["lp"] - t["m_ref"]) / ESCALA
t["precio_ok"] = (z_hat - t["z"]).abs() < 1e-6 * np.maximum(1, t["z"].abs())
REF_PRECIO = pd.DataFrame({"ocid": t["ocid"], "escala": ESCALA, "precio_ok": t["precio_ok"]}).set_index("ocid")
del t
df["escala_precio"] = df["ocid"].map(REF_PRECIO["escala"])
df["precio_ok"] = df["ocid"].map(REF_PRECIO["precio_ok"]).fillna(False).astype(bool)
say(f"\nPrecio reproducido en {100 * df['precio_ok'].mean():.2f} % de los procesos puntuables")

# constantes por grupo para las inyecciones (en la base completa, con la agrupación de cada regla)
def percentil_de_grupo(pct, q):
    regla = REGLAS[pct]
    x = num(DF_TODO[regla["fuente"]])
    k = clave_grupo(DF_TODO, regla["agrupacion"])
    return x.groupby(k).quantile(q)

P5_PLAZO = percentil_de_grupo("pct_plazo_corto", 0.05) if "pct_plazo_corto" in REGLAS else pd.Series(dtype=float)
P99_CAPITAL = percentil_de_grupo("pct_monto_capital", 0.99) if "pct_monto_capital" in REGLAS else pd.Series(dtype=float)
ANTIG_NUEVO = 30.0                                # días: proveedor constituido un mes antes
FACTORES = {"precio_x1,5": 1.5, "precio_x2": 2.0, "precio_x3": 3.0}
COMPETITIVAS = ("subasta", "menor cuantia", "cotizacion", "licitacion", "lista corta", "concurso")

def recetas_del_grupo(g, plan, filas24):
    """Para cada tipo de anomalía: procesos elegibles de 2024 y valores nuevos de X (escala de la base)."""
    usar = plan["usar"]
    sub = df.loc[filas24]
    R = {}
    if "precio_alto" in usar and "pct_precio_alto" in REGLAS:
        z = num(sub["ind_desv_precio_unit_z"])
        for nombre, k in FACTORES.items():
            z_nuevo = np.log(k) / sub["escala_precio"]
            ok = sub["precio_ok"] & z.notna() & (z < z_nuevo) & sub["escala_precio"].gt(0)
            # el indicador se recalcula desde el z nuevo (la fuente de pct_precio_alto es el z)
            filas = sub.index[ok]
            R[nombre] = (filas, {"precio_alto": recalcular("pct_precio_alto", filas, z_nuevo[ok].to_numpy())})
    if sin_tildes(g).startswith(COMPETITIVAS) and "oferente_unico" in usar and "ind_n_oferentes" in sub:
        filas = sub.index[num(sub["ind_n_oferentes"]) >= 2]
        cambios = {"oferente_unico": np.ones(len(filas))}
        if "poca_concurrencia" in usar:
            cambios["poca_concurrencia"] = np.ones(len(filas))
        R["oferente_unico"] = (filas, cambios)
    if "plazo_corto" in usar and "pct_plazo_corto" in REGLAS:
        regla = REGLAS["pct_plazo_corto"]
        p5 = clave_grupo(sub, regla["agrupacion"]).map(P5_PLAZO)
        actual = num(sub[regla["fuente"]])
        ok = actual.notna() & p5.notna() & (actual > p5)
        filas = sub.index[ok]
        R["plazo_p5"] = (filas, {"plazo_corto": recalcular("pct_plazo_corto", filas, p5[ok].to_numpy())})
    if {"proveedor_nuevo", "monto_capital"} <= set(usar) and {"pct_proveedor_nuevo", "pct_monto_capital"} <= set(REGLAS):
        r_an, r_mc = REGLAS["pct_proveedor_nuevo"], REGLAS["pct_monto_capital"]
        an, mc = num(sub[r_an["fuente"]]), num(sub[r_mc["fuente"]])
        p99 = clave_grupo(sub, r_mc["agrupacion"]).map(P99_CAPITAL)
        ok = an.notna() & (an > ANTIG_NUEVO) & mc.notna() & p99.notna() & (mc < p99)
        filas = sub.index[ok]
        cambios = {"proveedor_nuevo": recalcular("pct_proveedor_nuevo", filas, np.full(len(filas), ANTIG_NUEVO)),
                   "monto_capital": recalcular("pct_monto_capital", filas, p99[ok].to_numpy())}
        for v, val in [("debutante", 1.0), ("cuota_entidad", 0.0), ("recurrencia_par", 0.0), ("recompra_30d", 0.0)]:
            if v in usar:
                cambios[v] = np.full(len(filas), val)
        R["nuevo_monto_alto"] = (filas, cambios)
    # combinaciones: intersección de elegibles y unión de cambios
    for nombre, (a, b) in {"combinado_precio_oferente": ("precio_x2", "oferente_unico"),
                           "combinado_precio_plazo": ("precio_x2", "plazo_p5"),
                           "combinado_nuevo_precio": ("nuevo_monto_alto", "precio_x1,5")}.items():
        if a in R and b in R:
            filas = R[a][0].intersection(R[b][0])
            cambios = {}
            for parte in (a, b):
                f_p, c_p = R[parte]
                pos = f_p.get_indexer(filas)
                for v, arr in c_p.items():
                    cambios[v] = np.asarray(arr)[pos]
            R[nombre] = (filas, cambios)
    return {k: v for k, v in R.items() if len(v[0]) > 0}

H1_GRUPOS, RECETAS, filas_t = [], {}, []
for g, plan in PLAN.items():
    if not g.endswith("| pre"):
        continue
    filas24 = plan["idx"][(df.loc[plan["idx"], "anio_proc"] == 2024).to_numpy()]
    if len(filas24) < 1000:
        continue
    H1_GRUPOS.append(g)
    RECETAS[g] = recetas_del_grupo(g, plan, filas24)
    fila = dict(grupo=g, procesos_2024=len(filas24), inyecciones_por_semilla=int(round(CORTE_TOP * len(filas24))))
    fila.update({k: len(v[0]) for k, v in RECETAS[g].items()})
    filas_t.append(fila)
say("\nGrupos de H1 y procesos de 2024 elegibles para cada tipo de anomalía:")
say(pd.DataFrame(filas_t).fillna(0).set_index("grupo").T.to_string())
say(f"[celda 6: {time.time() - t_celda:.0f} s]")

In [ ]:
# =====================================================================================
# CELDA 7. H1: 20 semillas de inyección. En cada semilla y grupo se altera el 1 % de los procesos
#          de 2024, se reajustan los modelos sobre el grupo completo y se mide quién los encuentra.
#          Es la celda larga: guarda el avance después de cada semilla.
# =====================================================================================
empezar("celda 7")
t_celda = time.time()
SEMILLAS_H1 = list(range(1, 21))
METODOS = ["ensamble", "base", "if", "lof", "ae"]
CORTES = [0.01, 0.02, 0.05]
registros = []
archivo_avance = os.path.join(DIR_RUN, "h1_inyecciones.csv")
for s in SEMILLAS_H1:
    ts = time.time()
    for g in H1_GRUPOS:
        plan = PLAN[g]
        filas24 = plan["idx"][(df.loc[plan["idx"], "anio_proc"] == 2024).to_numpy()]
        rng = np.random.default_rng([s, H1_GRUPOS.index(g)])
        tipos = list(RECETAS[g])
        n_iny = max(1, int(round(CORTE_TOP * len(filas24))))
        orden = [tipos[i] for i in rng.permutation(len(tipos))]
        cuantos = pd.Series([orden[i % len(orden)] for i in range(n_iny)]).value_counts()
        Xg = X.loc[plan["idx"], plan["usar"]].copy()
        usados, marca = set(), pd.Series("", index=filas24)
        for tipo in orden:
            if tipo not in cuantos.index:
                continue
            filas_ok, cambios = RECETAS[g][tipo]
            libres = np.array([i for i in filas_ok if i not in usados])
            if len(libres) == 0:
                continue
            elegidos = rng.choice(libres, size=min(int(cuantos[tipo]), len(libres)), replace=False)
            pos = filas_ok.get_indexer(elegidos)
            for v, arr in cambios.items():
                Xg.loc[elegidos, v] = np.asarray(arr)[pos]
            usados.update(elegidos.tolist())
            marca.loc[elegidos] = tipo
        r, _, _ = puntuar_grupo(plan, Xg, SEMILLA + s)
        r24 = r.loc[filas24]
        iny = marca[marca != ""]
        for metodo in METODOS:
            pctl = r24[metodo].rank(pct=True)
            tops = {c: pd.Series(top_k(r24[metodo], c, SEMILLA + s), index=r24.index) for c in CORTES}
            for i, tipo in iny.items():
                registros.append(dict(semilla=s, grupo=g, proceso=int(i), tipo=tipo, metodo=metodo,
                                      percentil=float(pctl.loc[i]),
                                      **{f"en_top_{int(c * 100)}": bool(tops[c].loc[i]) for c in CORTES}))
    pd.DataFrame(registros).to_csv(archivo_avance, index=False)
    hechas = SEMILLAS_H1.index(s) + 1
    resta = (time.time() - t_celda) / hechas * (len(SEMILLAS_H1) - hechas) / 60
    say(f"  semilla {s:>2}: {sum(1 for x in registros if x['semilla'] == s and x['metodo'] == 'ensamble')} "
        f"inyecciones | {time.time() - ts:.0f} s | faltan unos {resta:.0f} min")
H1R = pd.DataFrame(registros)
say(f"[celda 7: {(time.time() - t_celda) / 60:.1f} min]")

In [ ]:
# =====================================================================================
# CELDA 8. Prueba de H1 tal como quedó en el preregistro y resultados secundarios
# =====================================================================================
empezar("celda 8")
par = H1R.pivot_table(index=["semilla", "grupo", "proceso", "tipo"], columns="metodo", values="en_top_1").astype(float)
par["d"] = par["ensamble"] - par["base"]
D = float(par["d"].mean())
por_semilla = par.groupby(level="semilla")["d"].agg(["sum", "count"])
rng_b = np.random.default_rng(SEMILLA)
B = 2000
boot = np.empty(B)
semillas = por_semilla.index.to_numpy()
for b in range(B):
    elegidas = rng_b.choice(semillas, size=len(semillas), replace=True)
    sel = por_semilla.loc[elegidas]
    boot[b] = sel["sum"].sum() / sel["count"].sum()
p_valor = float(np.mean(boot <= 0))
ic = np.percentile(boot, [2.5, 97.5])
cumple = p_valor < 0.05
rec = H1R[H1R["metodo"].isin(METODOS)].groupby("metodo")[["en_top_1", "en_top_2", "en_top_5"]].mean()

say("=" * 100)
say("H1 · RESULTADO PRIMARIO (preregistrado)")
say("=" * 100)
say(f"Inyecciones: {len(par):,} ({par.index.get_level_values('semilla').nunique()} semillas, "
    f"{len(H1_GRUPOS)} grupos)")
say(f"Recuperación en el 1 % superior: ensamble {par['ensamble'].mean():.4f} | índice simple {par['base'].mean():.4f}")
say(f"D = ensamble - índice simple = {D:+.4f} | IC 95 % (remuestreo de semillas) {ic[0]:+.4f} a {ic[1]:+.4f}")
p_txt = f"{p_valor:.4f}" if p_valor > 0 else f"< {1 / B:.4f} (ninguna de las {B} réplicas dio D <= 0)"
say(f"Valor p unilateral = {p_txt}  ->  H1 {'SE CUMPLE' if cumple else 'NO SE CUMPLE'} (regla: p < 0,05)")
# sensibilidad: remuestreo por inyección
boot_i = rng_b.choice(par["d"].to_numpy(), size=(B, len(par)), replace=True).mean(axis=1)
say(f"Sensibilidad (remuestreo por inyección): p = {float(np.mean(boot_i <= 0)):.4f}, "
    f"IC 95 % {np.percentile(boot_i, 2.5):+.4f} a {np.percentile(boot_i, 97.5):+.4f}")

say("\nSecundarios (descriptivos)")
say("Recuperación por método y corte (al azar: igual al corte):")
say(rec.round(4).to_string())
t_tipo = H1R.pivot_table(index="tipo", columns="metodo", values="en_top_1", aggfunc="mean")[METODOS]
t_tipo.insert(0, "inyecciones", H1R[H1R["metodo"] == "ensamble"].groupby("tipo").size())
say("\nRecuperación en el 1 % por tipo y magnitud:")
say(t_tipo.round(3).to_string())
t_grupo = H1R.pivot_table(index="grupo", columns="metodo", values="en_top_1", aggfunc="mean")[METODOS]
say("\nRecuperación en el 1 % por grupo:")
say(t_grupo.round(3).to_string())
say("\nPercentil mediano de los inyectados (1 = el más anómalo):")
say(H1R.pivot_table(index="tipo", columns="metodo", values="percentil", aggfunc="median")[METODOS].round(3).to_string())
say("\nD por semilla (estabilidad del resultado):")
say((por_semilla["sum"] / por_semilla["count"]).round(4).to_frame("D").T.to_string())

RESULTADO_H1 = dict(inyecciones=int(len(par)), semillas=SEMILLAS_H1, grupos=H1_GRUPOS,
                    recuperacion_ensamble=float(par["ensamble"].mean()), recuperacion_indice=float(par["base"].mean()),
                    D=D, ic95=[float(ic[0]), float(ic[1])], p_unilateral=p_valor, cumple=bool(cumple),
                    sensibilidad_por_inyeccion_p=float(np.mean(boot_i <= 0)))
H1R.to_csv(os.path.join(DIR_RUN, "h1_inyecciones.csv"), index=False)
t_tipo.to_csv(os.path.join(DIR_RUN, "h1_por_tipo.csv"))

In [ ]:
# =====================================================================================
# CELDA 9. Estabilidad de la lista con 20 semillas (descriptiva) + control de que la corrida
#          congelada se reproduce + registro final. Pon HACER_ESTABILIDAD = False para saltarla.
# =====================================================================================
empezar("celda 9")
HACER_ESTABILIDAD = True

def jaccard(a, b):
    a, b = np.asarray(a, bool), np.asarray(b, bool)
    u = (a | b).sum()
    return (a & b).sum() / u if u else np.nan

t_celda = time.time()
CONGELADA = os.path.join(SERCOP, "modelos", "20261016_1017_completo_normal_sup", "puntajes.parquet")
ESTAB20 = pd.DataFrame()
if HACER_ESTABILIDAD:
    c_ = duckdb.connect()
    ruta_c = CONGELADA.replace("\\", "/")
    cong = c_.execute(f"SELECT ocid, grupo, ensamble, top_ensamble FROM read_parquet('{ruta_c}')").df()
    cong = cong.set_index("ocid")
    filas_e = []
    for s in [SEMILLA] + list(range(1, 21)):
        for g, plan in PLAN.items():
            r2, _, _ = puntuar_grupo(plan, X.loc[plan["idx"]], s)
            r2 = marcar_tops(r2, s)
            oc = df.loc[r2.index, "ocid"].to_numpy()
            ref = cong.reindex(oc)
            j = jaccard(ref["top_ensamble"].fillna(False).to_numpy(bool), r2["top_ensamble"].to_numpy(bool))
            dif = float(np.nanmax(np.abs(ref["ensamble"].to_numpy(float) - r2["ensamble"].to_numpy(float))))
            filas_e.append(dict(semilla=s, grupo=g, jaccard=j, max_dif_puntaje=dif))
        say(f"  semilla {s}: Jaccard medio {np.mean([f['jaccard'] for f in filas_e if f['semilla'] == s]):.3f}"
            f"  ({(time.time() - t_celda) / 60:.1f} min)")
    ESTAB20 = pd.DataFrame(filas_e)
    rep = ESTAB20[ESTAB20["semilla"] == SEMILLA]
    say(f"\nControl de reproducción (semilla {SEMILLA}, la de la lista congelada): Jaccard mínimo "
        f"{rep['jaccard'].min():.3f}, diferencia máxima de puntaje {rep['max_dif_puntaje'].max():.2e}"
        + ("  (ok)" if rep["jaccard"].min() == 1 else "  <- la lista congelada no se reproduce exactamente"))
    est = ESTAB20[ESTAB20["semilla"] != SEMILLA]
    say("\nJaccard de la lista con las semillas 1 a 20 frente a la congelada (descriptivo):")
    say(est.groupby("grupo")["jaccard"].agg(["mean", "min", "max"]).round(3).to_string())
    ESTAB20.to_csv(os.path.join(DIR_RUN, "estabilidad_20_semillas.csv"), index=False)

registro = dict(corrida=RUN_ID, preregistro=FIRMA["huella_documento"], firma_fecha=FIRMA["fecha_firma"],
                huella_indicadores=HUELLA_IX, huella_procesos=HUELLA_PM, h1=RESULTADO_H1,
                estabilidad_media=float(ESTAB20.loc[ESTAB20["semilla"] != SEMILLA, "jaccard"].mean()) if len(ESTAB20) else None,
                versiones=dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,
                               sklearn=sklearn.__version__, duckdb=duckdb.__version__),
                minutos=round((time.time() - T_INICIO) / 60, 1))
with open(os.path.join(DIR_RUN, "registro_h1.json"), "w", encoding="utf-8") as f:
    json.dump(registro, f, ensure_ascii=False, indent=1, default=float)
say("\n" + "=" * 100)
say(f"H1: {'SE CUMPLE' if RESULTADO_H1['cumple'] else 'NO SE CUMPLE'} | D {RESULTADO_H1['D']:+.4f} | "
    f"p {RESULTADO_H1['p_unilateral']:.4f} | {registro['minutos']} min")
say("Archivos en", DIR_RUN)
guardar_txt()
say("\nAdjunta en el chat el archivo resumen_para_chat.txt de esa carpeta.")
guardar_txt()